## Aim:
How to link the rule-based extracted CI_TYPE-GEO pairs with the respective Ci failure impacts
 
**Idea**\
Test using prompt engineering by passing table of CI-GEO pairs to GPT-J model.
Steps:
* Load model and apply it always on one chunk of the document to extract CI failure impacts 
* Use prompt engineering to extract time and location of the CI failure (origin) the CI impacts (impact location)
or 
* Pass dataframe of pairs as input to the model
or
* Use few shot prompting with example answers

**Finally:**
* Compaire all approaches of spatial and temporal linking CI failure impacts


In [ ]:
%load_ext scalene


In [ ]:
%%scalene --reduced-profile    # --cpu-only --cpu-sampling-rate 0.0001
# kernel crashes


In [6]:

#
import os


# # settings for CUDA and PYTORCH
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # settings for debugging CUDA errors (pinpoint exact line of error)
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 


# ### settings for distributed computing
# import torch
# from torch import distributed as dist
# from transformers import AutoTokenizer, AutoModelForCausalLM

# # NOTE find MASTER_ADDR and MASTER_PORT  (any unused port)
# # hostname -I | awk '{print $1}'
# # comm -23 <(seq 29500 30000 | sort) <(ss -Htan | awk '{print $4}' | cut -d: -f2 | sort -u) | head -n 1
# os.environ["WORLD_SIZE"]="1"
# os.environ["RANK"]="0"
# os.environ["LOCAL_RANK"]="0"
# os.environ["MASTER_ADDR"]="10.10.0.2" # "localhost"
# os.environ["MASTER_PORT"]="29500" #"12355" #

# local_rank = int(os.environ["LOCAL_RANK"])
# torch.cuda.set_device(local_rank)
# dist.init_process_group(backend="nccl")


# activate global venv explicitly
os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())  # should give 2
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


## --> must be CUDA 12.6, torch: 2.91, ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']

import os
import sys
# import subprocess
import re
import time
from glob import glob
from pathlib import Path
import gc
from typing import List, Dict, Tuple, Optional, Union, TypeVar
# from io import StringIO
import json

# from tqdm import tqdm
import numpy as np
import pandas as pd
from fuzzywuzzy import fuzz
import geonamescache
# import pyarrow as pa
# import pyarrow.parquet as pq
import spacy
from huggingface_hub import login
from pdfminer.high_level import extract_text
import langdetect
from transformers import AutoTokenizer
from haystack.dataclasses import ByteStream
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types import DoclingDocument
from docling_core.types.doc import DocItemLabel
from docling_core.types.doc.document import SectionHeaderItem, ListItem, TextItem, DocItem
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
# from langchain.document_loaders import DirectoryLoader #, UnstructuredLoader
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredLoader
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.document_converter import DocumentConverter, FormatOption
from docling.chunking import HybridChunker



from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
import src.extraction_model as em
import src.postprocess as pp
import src.utils as u
from geollama.geollama.main import GeoLlama
from geollama.geollama.model import TopoModel, RAGModel



test_mode = True

# login to HF
# login(token=os.getenv("HUGGINGFACE_TOKEN"))
try: 
    login(token=os.getenv("HUGGINGFACE_TOKEN"))   # notebook_login
except:
    login(token=os.environ.get("HUGGINGFACE_TOKEN"))  # former HF_TOKEN
        
# NOTE raises exception if not env.variable doesnt exist (compared to os.envrion.get and its shortcut os.getenv)


# NOTE. disabled batch size as OOM for CUDA despite chunkwise memory cleaning, nvtop to find best batchsize
BATCH_SIZE = s.BATCH_SIZE  # max for nvidia GPU

# torch.manual_seed(42)

#  automatic linebreaks and multi-line cells.
pd.set_option('display.max_colwidth', 100000)
pd.set_option("display.colheader_justify", "left")

print(os.environ["CUDA_VISIBLE_DEVICES"])

# clean up before applying CUDA
# gc.collect()
# torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()


### LOGGER

# currently fetch only PydanticSerializationUnexpectedValue warnings 
# See, discussion in https://github.com/openai/openai-python/issues/2872
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")
# # Or targeted:
# import logging
# logging.getLogger("pydantic").setLevel(logging.ERROR)


# ## TODO test to prevent CUDA-OOM when reused
# ## Source: https://spacy.io/usage/embeddings-transformers
# from thinc.api import set_gpu_allocator, require_gpu

# # Use the GPU, with memory allocations directed via PyTorch.
# # This prevents out-of-memory errors that would otherwise occur from competing
# # memory pools.
# set_gpu_allocator("pytorch")
## require_gpu(0)



# %% [markdown]
# ## Set paths and vars

# %%
# set wd to project root
# os.chdir("/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval")

## set path variables
DOCS_DIR = Path(s.PATH_DATA / "text_sources/")
PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")
NER_PATTERNS_FILEPATH = Path(s.NER_PATTERNS_FILEPATH)
LLM_OUTPUTS_DIR = Path(s.PATH_DATA / "llm_outputs/")
geollama_OUTPUTS_DIR = Path(s.PATH_DATA / "geollama_outputs/")

os.makedirs(PARSED_TEXT_DIR, exist_ok=True)
os.makedirs(s.PATH_LLM_DATA, exist_ok=True)
os.makedirs(geollama_OUTPUTS_DIR, exist_ok=True)


# CI GEO pairs
CI_GEO_FILEPATH = Path( s.PATH_DATA / s.CI_GEO_PAIRS_FILENAME)

## store LLM 1 response and prompt
OUTPUT_LLM1_FILEPATH =  Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
OUTPUT_geollama_FILEPATH =  Path(s.PATH_LLM_DATA / "geollama_results.csv")




# %% [markdown]
# ### Set test mode

# %%

docs_list_sample = [


# Path(PARSED_TEXT_DIR, "Khazai 2013 - Juni-Hochwasser 2013 in Mitteleuropa - Fokus Deutschland Bericht 2 Auswirkungen und Bewältigung.md"),
# Path(PARSED_TEXT_DIR, "Rozendaal 2021 - Infrabel_ Flood damage to railway track worth tens of millions of euros _ SpoorPro - incomplete.md"),
Path(PARSED_TEXT_DIR, "Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent.md"),
#         Path(PARSED_TEXT_DIR, "Chamra 2006 - Flooding of the Prague metro during the August 2002 floods.md"),
#         # Path(PARSED_TEXT_DIR, "Fink 2009 - The European storm Kyrill in January 2007_ synoptic evolution, meteorological impacts and some considerations with respect to climate change.md"),
#         # Path(PARSED_TEXT_DIR, "Hladny 2004 - August_2002_catastrophic_flood_in_the_Czech_Republic.md"),
#         # Path(PARSED_TEXT_DIR, "Mitsakis 2014 - Impacts of high-intensity storms on urban transportation_ applying traffic flow control methodologies for quantifying the effects.md"),
#         # Path(PARSED_TEXT_DIR, "Pescaroli 2017 - How Critical Infrastructure Orients International Relief in Cascading.md"),
    
        Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods.md"),
        Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping.md"), 
        Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga.md"),

        Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece.md"),
        Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia.md"),
        Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News.md"),     
            Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt.md"),
            Path(PARSED_TEXT_DIR, "AFP 2022 - The_Vibes_Valencia Airport in Madrid briefly shut as lightning hits runway _ World _ The Vibes.md"),
            Path(PARSED_TEXT_DIR, "Artemis 2015 - PERILS finalises Storm Desmond UK flood loss estimate at £604m.md"),
            Path(PARSED_TEXT_DIR, "Brown 2010 - Economy feels chill as UK grinds to a halt _ The Independent.md"),
        #     # Path(PARSED_TEXT_DIR, "Diakakis 2020 - A systematic assessment of the effects of extreme flash floods on transportation infrastructure and circulation: The example of the 2017 Mandra flood.md"),
        Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158.md"),
        #     # Path(PARSED_TEXT_DIR, "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids.md"),
Path(PARSED_TEXT_DIR, "Euronews 2024 - Spain floods_ Death toll rises to 205 as nation braces for more rain .md"),
Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade.md"),
          Path(PARSED_TEXT_DIR, "Fink 2004 - The 2003 European summer heatwaves and drought - synoptic diagnosis and impacts.md"),
          Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News.md"),
          #     # Path(PARSED_TEXT_DIR, "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study.md"),
          Path(PARSED_TEXT_DIR, "Kaur 2025 - Authorities suspect arson in 17 wildfires across Dalmatian coast, Croatia - The Watchers.md"),
          #     # Path(PARSED_TEXT_DIR, "Keller 2014 - Mapping Natural Hazard Impacts on Road Infrastructure—The Extreme Precipitation in Baden-Württemberg.md"),
            # Path(PARSED_TEXT_DIR, "Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding.md"),
            # Path(PARSED_TEXT_DIR, "Korzilius 2021 Nach der Flut.md"),

    Path(PARSED_TEXT_DIR, "Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events.md"),

    Path(PARSED_TEXT_DIR, "Chamra 2006 - Flooding of the Prague metro during the August 2002 floods.md"),
    Path(PARSED_TEXT_DIR, "Hladny 2004 - August_2002_catastrophic_flood_in_the_Czech_Republic.md"),
    Path(PARSED_TEXT_DIR, "Mitsakis 2014 - Impacts of high-intensity storms on urban transportation_ applying traffic flow control methodologies for quantifying the effects.md"),
    Path(PARSED_TEXT_DIR, "Pescaroli 2017 - How Critical Infrastructure Orients International Relief in Cascading.md"),
    Path(PARSED_TEXT_DIR, "Nieuwsblad 2021 - Geen drinkbaar water, geen gas, en geen elektriciteit_ 9.000 mensen moeten op zoek naar ander onderkomen.md"),

    Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication.md"),
    Path(PARSED_TEXT_DIR, "Treanor 2015 - Storm Desmond damage across Cumbria estimated at £500m _ Storm Desmond _The Guardian.md"),
    Path(PARSED_TEXT_DIR, "Nour 2011 - Damages caused by floods and flash-floods upon critical infrastructure.md"),
    Path(PARSED_TEXT_DIR, "Fink 2009 - The European storm Kyrill in January 2007_ synoptic evolution, meteorological impacts and some considerations with respect to climate change.md"),
    
    # # # long processing
    Path(PARSED_TEXT_DIR, "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS.md"),
    # Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication.md"),
    

    #     #     # not part of valid set:
    #     #     # Path(PARSED_TEXT_DIR, "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events.md"), # > 1800 entries LLMv3.0 incl. hallucinations
]




## Test mode
if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))

print(f"Using {len(search_path)} documents for processing.")



# %% [markdown]
# ###  Load spaCy language model

## RELOAD spacy pipeline
nlp = spacy.load("./spacy_model_pipeline")

# add CI_TYPE patterns to spacy nlp model pipeline
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk(NER_PATTERNS_FILEPATH)
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk(NER_PATTERNS_FILEPATH)


# load NER patterns for CI types and their subgroups (needed for cleaning LLm response - STEP 1 before continuing with STEP 2)
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

# %%


# %% [markdown]
# ### geollama pipeline


# %%
## Make sure that still both GPUS are visible

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# # !nvidia-smi


topo_model = TopoModel(
    model_name='JoeShingleton/GeoLlama-3.2-3b-toponym',
    # model_name='JoeShingleton/GeoLlama_7b_toponym', 
    prompt_path='../geollama/data/prompt_templates/prompt_template.txt',
    instruct_path='../geollama/data/prompt_templates/topo_instruction.txt',
    input_path=None,
    config_path='../geollama/data/config_files/model_config.json'
)

rag_model = RAGModel(
    model_name='JoeShingleton/GeoLlama-3.2-3b-RAG',
    # model_name='JoeShingleton/GeoLlama_7b_RAG',
    prompt_path='../geollama/data/prompt_templates/prompt_template.txt',
    instruct_path='../geollama/data/prompt_templates/rag_instruction.txt',
    input_path='../geollama/data/prompt_templates/rag_input.txt',
    config_path='../geollama/data/config_files/model_config.json')

geo_llama = GeoLlama(
    topo_model = topo_model, 
    rag_model = rag_model, 
    translate_model=None
)


0
True
1
NVIDIA A100-PCIE-40GB
_CudaDeviceProperties(name='NVIDIA A100-PCIE-40GB', major=8, minor=0, total_memory=40441MB, multi_processor_count=108, uuid=49cb0d59-8657-8b70-8dd2-c1db7aab24a4, pci_bus_id=131, pci_device_id=0, pci_domain_id=0, L2_cache_size=40MB)
(8, 0)
['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
2.8.0+cu128
12.8


0
6.19708416
Test mode is ON. Using only a small sample of documents for testing.
Using 28 documents for processing.
Using flash attention


Using locally saved model


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

NOTE - loading tokenizer not from fine-tuned geollama model, but from its anchestor: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Using flash attention


Using locally saved model


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

NOTE - loading tokenizer not from fine-tuned geollama model, but from its anchestor: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [7]:
print(os.environ["CUDA_VISIBLE_DEVICES"])
#os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
print(os.environ["CUDA_VISIBLE_DEVICES"])


0
0


In [3]:
# # # # settings for CUDA and PYTORCH
# # os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# # print(os.environ["CUDA_VISIBLE_DEVICES"])
# # # os.environ["CUDA_VISIBLE_DEVICES"]="0"
# # os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # # settings for debugging CUDA errors (pinpoint exact line of error)
# # os.environ["TORCH_USE_CUDA_DSA"] = "1"
# # # os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# # # activate global venv explicitly
# # os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"

# import os 
# import torch

# print(torch.cuda.is_available())
# print(torch.cuda.device_count())  # should give 2
# print(torch.cuda.get_device_name())
# print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
# print(torch.cuda.get_device_capability())
# print(torch.cuda.get_arch_list())
# print(torch.__version__)
# print(torch.version.cuda)
# # has two gpus nabled? 


# # # # settings for CUDA and PYTORCH
# # # os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# # print(os.environ["CUDA_VISIBLE_DEVICES"])
# # os.environ["TORCH_USE_CUDA_DSA"] = "1"

# # # os.environ["CUDA_VISIBLE_DEVICES"]="0"
# # # # os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # # # settings for debugging CUDA errors (pinpoint exact line of error)
# # # os.environ["TORCH_USE_CUDA_DSA"] = "1"
# # # # os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# print(os.environ["CUDA_VISIBLE_DEVICES"])

# # # settings for CUDA and PYTORCH
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # settings for debugging CUDA errors (pinpoint exact line of error)
# os.environ["TORCH_USE_CUDA_DSA"] = "1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# print(torch.cuda.is_available())
# print(torch.cuda.device_count())
# # %%
# # Settings
# #model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "meta-llama/Meta-Llama-3-70B-Instruct"


# from huggingface_hub import login
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     BitsAndBytesConfig,
#     DynamicCache,
# )

# from src.settings import settings as s
# import src.utils as u


# model_dir = s.HF_HOME_DIR   # use default dir in .cache/
        
# print(f"Model directory: {model_dir}")

# # quantization config
# # Load model with 4-bit quantization if applicable (use 4-bit integer instead of 32b floats) --> reduce the required VRAM for model application
# # see, https://huggingface.co/docs/transformers/quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )
# flash_attn_config = "flash_attention_2"

# model = AutoModelForCausalLM.from_pretrained(
#         model_name,
#         cache_dir=model_dir,
#         local_files_only=True,  # tp_plan="auto" # set tensor parallel model (ie. splits model on multiple GPU)
#         # max_seq_length=2048,
#         dtype="auto", # None ,# test for CU12.6, torch.29.1 #"auto",
#         device_map="auto",
#         attn_implementation=flash_attn_config,
#         quantization_config=bnb_config,
#         # tp_plan="auto",  # automatically use a tensor parallelism plan based on predefined configuration of the model (i.e. partition model on both GPUs)
#     )
# print(model.hf_device_map)
# {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 0, 'model.layers.26': 0, 'model.layers.27': 0, 'model.layers.28': 0, 'model.layers.29': 0, 'model.layers.30': 0, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.layers.40': 1, 'model.layers.41': 1, 'model.layers.42': 1, 'model.layers.43': 1, 'model.layers.44': 1, 'model.layers.45': 1, 'model.layers.46': 1, 'model.layers.47': 1, 'model.layers.48': 1, 'model.layers.49': 1, 'model.layers.50': 1, 'model.layers.51': 1, 'model.layers.52': 1, 'model.layers.53': 1, 'model.layers.54': 1, 'model.layers.55': 1, 'model.layers.56': 1, 'model.layers.57': 1, 'model.layers.58': 1, 'model.layers.59': 1, 'model.layers.60': 1, 'model.layers.61': 1, 'model.layers.62': 1, 'model.layers.63': 1, 'model.layers.64': 1, 'model.layers.65': 1, 'model.layers.66': 1, 'model.layers.67': 1, 'model.layers.68': 1, 'model.layers.69': 1, 'model.layers.70': 1, 'model.layers.71': 1, 'model.layers.72': 1, 'model.layers.73': 1, 'model.layers.74': 1, 'model.layers.75': 1, 'model.layers.76': 1, 'model.layers.77': 1, 'model.layers.78': 1, 'model.layers.79': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## Document cleaning

In [32]:
# # load tokenizer
# embed_model =  "sentence-transformers/all-MiniLM-L6-v2"
# tokenizer = HuggingFaceTokenizer(
#     tokenizer=AutoTokenizer.from_pretrained(embed_model),
#     max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
#     # standardize input sizes of chunks for Llama models
#     padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
#     truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
# )

# ## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
# chunker = HybridChunker(
#     tokenizer=tokenizer,
#         # max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
#         # chunk_overlap=0, # no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
#     split_by_sentence=True, # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
#     merge_peers=True,  # optional, defaults to True
# )



In [33]:

# # print( "Number of documents to process:", len(os.listdir(DOCS_DIR)) )
# print("Number of documents to process:", len(search_path) )
# start_time = time.time()


# # EXPORT_TYPE = ExportType.DOC_CHUNKS
# #     mislabeled: List[DocItem] = []
# #     min_paragraph_size = 100  
# #     temp_docs: List[ByteStream] = []
# #     temp_meta: List[Dict[str, str]] = []
# #     i: int
# #     combined_paragraph: str = ""
# #     combined_chars: int = 0
# #     para_num: int = 0
# #     section_name: str = ""
# #     page_no: Optional[int] = None
# #     first_note: bool = False




# # convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc

# for filename in os.listdir(DOCS_DIR):

#     if filename.endswith(".pdf"):

#         pdf_filepath = os.path.join(DOCS_DIR, Path(filename))

#         md_filename = f"{Path(filename).stem}.md"
#         md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
#         cleaned_jsonl_filepath = md_filepath.replace(".md", "_cleaned.jsonl")
#         cleaned_md_filepath = md_filepath.replace(".md", "_cleaned.md")

#         if os.path.exists(cleaned_jsonl_filepath):
#             print(f"Cleaned markdown file already exists: '{cleaned_md_filepath}'")
#             continue
        

#         print(f"\nFetching: {filename}")

#         # get language of document
#         src_language_doc = langdetect.detect(filename.replace(".pdf", "").lower())  # lower case improves language detection


#         ## Document converter with OCR
#         print("Using OCR for text extraction as it identifies section titles, footers/headers and pagenumbers as such, but reads in also figure text sometimes") 
#         # NOTE all other standard doclingConverter retunr section/headers etc as BODY not FURNITURE
#         # NOTE: partly reads in figure text and table text 
#         pdf_doc_org = dc.DocumentParser().ocr_converter.convert(source=pdf_filepath).document  ## !! recognizes section titles, footers/headers !! :D

#         ##  get only list of Doc.items
#         texts = dc.get_processed_texts(pdf_doc_org) 

#         texts_clean = []
#         section_names = []
#         for i, text in enumerate(texts):
            

#  ### as Parser Doc class
#             # get next text only when it is not page header/footer
#             next_text = dc.get_next_text(texts, i)
#             # page_no = get_current_page(text, combined_paragraph, page_no)


#             # Update section header if the element is a section header
#             # TODO: Need a stronger check on section headers that takes top of page into account, etc
#             if dc.is_section_header(text) and text not in mislabeled:
#                 print("!!  Section header found:", text.text)
#                 section_name = text.text
#                 continue

#             if dc.is_reference_section(section_name):
#                 print("Reference section found. Stopping further processing of document.")
#                 break  

#             if dc.should_skip_element(text):
#                 continue
            
#             # clean from double whitespace, newlines, etc.
#             p_str = dc.clean_text(text.text)

#             # clean from potential figure references
#             p_str = dc.remove_figure_references(p_str)

#             ## replace e.g. and i.e. --> eg and ie to avoid sentence splits
#             p_str = re.sub(r"e\.g\.\s+", "eg ", p_str)
#             p_str = re.sub(r"i\.e\.\s+", "ie ", p_str)

#             # Removing URLs 
#             # LangExtract tries to open these URLs when they occur in the document text
#             # p_str= re.sub(r"http\S+", "", p_str) 

#             p_str_chars = len(p_str)

#             # If the paragraph does not end with final punctuation, accumulate it
#             if not dc.is_sentence_end(p_str):
#                 combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
#                 combined_chars += p_str_chars
#                 continue

#             # p_str ends with a sentence end; decide whether to process or accumulate it
#             total_chars = combined_chars + p_str_chars
#             if dc.is_section_header(next_text):
#                 # Immediately process if the next text is a section header
#                 p_str = dc.combine_paragraphs(combined_paragraph, p_str)
#                 combined_paragraph, combined_chars = "", 0
#             elif total_chars < min_paragraph_size:
#                 # Not enough characters accumulated yet; decide based on next_text
#                 if next_text is None or (not dc.is_page_text(next_text) and dc.is_sentence_end(p_str)):
#                     # End of document or next text item is not a text item and current paragraph ends with punctuation
#                     # Process the paragraph and reset the accumulator even though this is a short paragraph
#                     p_str = dc.combine_paragraphs(combined_paragraph, p_str)
#                     combined_paragraph, combined_chars = "", 0
#                 else:
#                     # Combine with next paragraph
#                     combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
#                     combined_chars = total_chars
#                     continue
#             else:
#                 # Sufficient characters: process the paragraph and reset the accumulator
#                 p_str = dc.combine_paragraphs(combined_paragraph, p_str)
#                 combined_paragraph, combined_chars = "", 0

#             p_str = dc.combine_hyphenated_words(p_str)
#             if p_str:  # Only add non-empty content
#                 para_num += 1
#                 dc.add_paragraph(
#                     p_str, 
#                     #para_num, section_name, page_no, 
#                     temp_docs, 
#                     # temp_meta
#                 )
#                 page_no = None
            
#             print("\n-__ Paragraph #", para_num, "; Section:", section_name)

#             # print(p_str)
#             # print(temp_docs, temp_meta)

# ##### as CLASS DCOPAraser=
# ### translator class
#             ## Translation
#             if src_language_doc != "en":

#                 supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
#                 # supported_languages = ["en", "fr", "de", "es", "pt", "it", "pl", "cs", "nl", "da", "sv", "no", "hr", "ro", "bg", "sl", "sk", "lt", "et" ],
#                 if src_language_doc not in supported_languages:
#                     print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
#                     continue 

#                 print(f"Translating {src_language_doc} --> en")
#                 p_str = tm.translate_2_english(src_language_doc, p_str)     
                
#             # collect cleaned text + meta data per doc            
#             texts_clean.append(p_str)
#             section_names.append(section_name)


#         print("TEST Saving cleaned text as DoclingDocument to store text (p_str)+ meta (sectiontitle, page_no): ")
#         # Initialize new DoclingDoc
#         doclingdoc = DoclingDocument(schema_name="DoclingDocument",  version="1.0.0",name="My Custom Document")
        
#         # write text and structural elements (titles, sections, and paragraphs) to doc     
#         title_node = doclingdoc.add_title(text=f"{filename.replace('.pdf', '')}")
#         for text, section_name in zip(texts_clean, section_names):
#             print(section_name, ":", text)
#             section_node = doclingdoc.add_heading(text=section_name, level=1, parent=title_node)
#             doclingdoc.add_text(label=DocItemLabel.TEXT, text=text, parent=section_node)

#         # Save doclingDocument as MD
#         doclingdoc.save_as_markdown(cleaned_md_filepath)     


#        ## STEP 2: then do chunking with HybridChunker  TODO
#         # # NOTE maybe only a bit needed as already very good splits but maybe tokensizes need to be adapted 
#         ## TODO inlcude p_str always as doclingobj part or write it back to DoclingObject, (maybe with section_name as meta info)

#         # chunking
#         chunk_iter = chunker.chunk(dl_doc=doclingdoc)
#         chunks = list(chunk_iter)

#         # # apply contextualization
#         # ser_text = chunker.contextualize(chunk=chunk)
#         # ser_tokens = tokenizer.count_tokens(ser_text)
#         # print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_text!r}")
#         # print()

#         break


# end_time = time.time() - start_time
# print(f"Parsing and cleaning done. Time elapsed: {end_time:.2f} seconds.")


# # visual check of removed items
# # TODO make as document_cleaning function: print removed items with largest number of chars first
# # ## NOTE. high number of chars == more potentially actual text body

# # text_items_removed = sorted(text_items_to_drop_visualization, key=lambda x: -x[0])
# # for i in text_items_removed[:50]:
# #     print(i) # -->  also subsection titles were removed partly



### bump from transformers to Langchain + Memory passing


In [ ]:

# class Memory:
#     # Source. https://medium.com/@jagadeesan.ganesh/mastering-llm-ai-agents-building-and-using-ai-agents-in-python-with-real-world-use-cases-c578eb640e35
#     def __init__(self):
#         self.memory_store = []

#     def remember(self, interaction):
#         self.memory_store.append(interaction)

#     def recall(self):
#         return " ".join(self.memory_store)
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.prompts import PromptTemplate

# model_name = "meta-llama/Llama-3.1-8B-Instruct"


# # prompt_template = "Tell me a {adjective} joke"
# # prompt = PromptTemplate(input_variables=["adjective"], template=prompt_template)
# model = AutoModelForCausalLM.from_pretrained(
#                 model_name,
#                 dtype="auto", # None ,# test for CU12.6, torch.29.1 #"auto",
#                 # max_memory={0: "2GB", 1: "10GB"},  # distribute memory across GPUs
#             )
# # chain = prompt | model #| StrOutputParser()

# # chain.invoke("your adjective here")


# # Define a simple prompt for the agent
# template = """
# You are an AI assistant with expertise in data analysis and automation. Answer the following question:
# Question: {question}
# """

# # Set up the prompt and LLM chain
# prompt = PromptTemplate(template=template, input_variables=["question"])
# chain = prompt | model | StrOutputParser() # LLMChain(prompt=prompt, llm=llm)

# # # # Example query
# # query = "What is the impact of AI in healthcare?"
# # response = chain.run(question=query)
# # print(f"Agent Response: {response}")


# from langchain.agents import create_agent

# agent = create_agent(
#     model=model_name,
#     model_provider="huggingface",
#     # tools=[get_weather],
#     system_prompt="You are a helpful assistant",
#     temperature=0.0,
#     max_tokens=1024,
# )

# result = agent.invoke(
#     {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
# )
# print(result["messages"][-1].content_blocks)

### Guide quantification of impacts

Test SPACY identify money (5,000 million, 5.000 billion, <pound>100.000, 100$, 100€), number of people affected, kms, tonnes etc. 
RECHECK NUMBERS against LIST of Pot. (NER) CANIDATES [1$, <PoundSymbol>100.000 million,A-3, A3 ,B 575] occurd in chunk:

* SKU-441 contains 441 — that is a label, not a price 
* [RAG Hallucinates — I Built a Self-Healing Layer That Fixes It in Real Time | Towards Data Science] - 
* MAYBE add this FUNC as INPUT to LLM (as TOOL, not as Candi.list in Prompt)

In [ ]:
# def _extract_numbers(text: str) -> set[str]: 
#     # SKU-441 → excluded (letter+hyphen prefix means it's a label) # 5-7 days → preserved (digit+hyphen is a numeric range) # $49.99 → preserved r
#     # Without this rule, every product code containing a number triggers a false positive. 
#     # return set(re.findall( r'(?<![A-Za-z]\-)\b\d+(?:\.\d+)?(?:%|k|m|b)?\b(?!\-[A-Za-z])', text.lower() )) 
#     # return sorted(set(re.findall(r'\b\d+(?:\.\d+)?(?:%|k|m)?\b', text.lower())))

In [ ]:
test = """
    The product SKU-441 costs $49.99 and will be delivered in 5-7 days. The discount is 20% and the stock is 1k units.
    The flood caused damages of around 2.5 million dollars (2.5$m), 2.5$ m,  
    affecting approximately 5,000 people and 300 businesses. 
    The recovery process is expected to take 6-12 months, 
    with an estimated cost of $1.2 billion. 
    The government has allocated a budget of $500 million for immediate relief efforts, 
    while the remaining funds will be used for long-term reconstruction and support for affected communities.
    Around 15 or 16 schools were closed
                 """


##  CI impact extraction


##### test if other qunatization maybe works for GPT-OSS

In [ ]:
# ## # settings for distributed computing
# # NOTE find MASTER_ADDR and MASTER_PORT  (any unused port)
# # hostname -I | awk '{print $1}'
# # comm -23 <(seq 29500 30000 | sort) <(ss -Htan | awk '{print $4}' | cut -d: -f2 | sort -u) | head -n 1
# import os
# os.environ["WORLD_SIZE"]="1"
# os.environ["RANK"]="0"
# os.environ["LOCAL_RANK"]="0"
# os.environ["MASTER_ADDR"]="localhost" # 10.10.0.2
# os.environ["MASTER_PORT"]="29500" # 12355

# import torch
# from torch import distributed as dist
# # from transformers import AutoTokenizer, AutoModelForCausalLM

# local_rank = int(os.environ["LOCAL_RANK"])
# torch.cuda.set_device(local_rank)
# dist.init_process_group(backend="nccl")


# from transformers import AutoModelForCausalLM, AutoTokenizer
# from transformers.distributed import DistributedConfig
# from transformers import BitsAndBytesConfig
# from pathlib import Path

# model_name = "openai/gpt-oss-20b"
# #model_name = "meta-llama/Llama-3.1-8B-Instruct"

# HF_HOME_DIR = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub"

# base_dir = HF_HOME_DIR   # use default dir in .cache/
# model_dir = Path(base_dir) 


# # tokenizer = AutoTokenizer.from_pretrained(model_name)
# # model = AutoModelForCausalLM.from_pretrained(
# #     model_name,
# #     device_map="auto",
# #     torch_dtype="auto",
# #     # Optimize MoE layers with downloadable` MegaBlocksMoeMLP
# #     use_kernels=True,
# # )

# # NOTE distributed loading and computational parallelism needed for GPT-OSS-20B
# # device_map: model loaded on memory across GPUs ;  TP focuses on computational parallelism
# device_map = {
#             # Enable Expert Parallelism
#             "distributed_config": DistributedConfig(enable_expert_parallel=1),
#             # Enable Tensor Parallelism
#             "tp_plan": "auto",
#         } 
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     # Optimize MoE layers with downloadable` MegaBlocksMoeMLP
#     use_kernels=True,
#     **device_map
# )
# tokenizer = AutoTokenizer.from_pretrained(model_name, **device_map,)

# # model = AutoModelForCausalLM.from_pretrained(
# #     model_name,
# #     # cache_dir=model_dir,
# #     #local_files_only=True, 
# #     dtype="auto",  # torch.float32
# #     attn_implementation="kernels-community/vllm-flash-attn3",
# #     use_kernels=True,
# #     **device_map,
# # )
# # tokenizer = AutoTokenizer.from_pretrained(
# #     model_name,
# #     use_fast=True,
# #     # device_map="auto",
# #     # cache_dir=model_dir,  # use fast Rust-based tokenizer, when possible
#     # **device_map
# # )

# messages = [
#     {"role": "user", "content": "How many rs are in the word 'strawberry'?"},
# ]

# inputs = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_tensors="pt",
#     return_dict=True,
# ).to(model.device)

# generated = model.generate(**inputs, max_new_tokens=100)
# print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))



In [ ]:
# In the word'strawberry', there are 2 Rs.<|eot_id|>

# generated

#### GPT-OSS  + tool for NER test

In [8]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""Data mining model (decoder, tokenizer) for extracting impacts on infrastructure"""

__author__ = "Anna Buch, TU Berlin"
__email__ = "anna.buch@tu-berlin.de"



import os
import copy
import json 
from typing import Optional, List
from pathlib import Path
from jinja2 import Environment, FileSystemLoader

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DynamicCache,
)
from transformers.distributed import DistributedConfig
from openai import OpenAI
    
import torch

from src.settings import settings as s
import src.utils as u
from src import postprocess as pp
        
import spacy



class NERModel():

    def __init__(self):
        self.nlp_model = self.initialize()
    

    def initialize(self):
        """ Return initialized NER model"""
  
        dir_model_pipeline="./spacy_model_pipeline"
        dir_ci_patterns=Path("./ner_patterns.jsonl")
        
        # LOAD spacy pipeline
        nlp = spacy.load(dir_model_pipeline)

        # add CI_TYPE patterns to spacy nlp model pipeline
        config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
        ## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
        ## NOTE EntityRuler is hidden inside .add_pipe()
        try:
            ruler = nlp.add_pipe("span_ruler", config=config)
            ruler.from_disk(dir_ci_patterns)
        except ValueError:
            print("SpanRuler already exists in pipeline.")
            ruler = nlp.get_pipe("span_ruler")
            ruler.from_disk(dir_ci_patterns)

        # load NER patterns for CI types and their subgroups (needed for cleaning LLm response - STEP 1 before continuing with STEP 2)
        ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

        return nlp



class ResponseItems(BaseModel):
    infrastructure_type: str
    damage: Optional[str] = None
    damage_value: Optional[str | List[str] | int | float] = None 
    location: str
    location_type: Optional[str] = None

class Response(BaseModel):
    impact_cases: List[ResponseItems]
    


def load_prompt_template(
    template_path: str = "./prompt_templates",
    template_filename: str = None,
):
    env = Environment(loader=FileSystemLoader(template_path))
    template = env.get_template(template_filename)

    return template



class Tools():  
    
    def __init__(self):
        
        self.tools = [
            {
                "type": "function",
                "name": "get_infrastructure_entities",
                "strict": True,
                # "description": "Get the entities about critical infrastructure (ci) assets from the text by using Named Entity Recognition (NER) from spaCy and an customized set of keywords (ci_keywords)",
                "description": "Get the entities about critical infrastructure (ci) assets from the text by using Named Entity Recognition (NER) from spaCy",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "ci_entity": {
                            "type": "string" , # List[str],  # string
                            "description": "The critical infrastructure (ci) entities from the given text",
                        },
                    "required": ["ci_entity"],
                    "additionalProperties": False,
            },},},
        # {
        #         "type": "function",
        #         "name": "get_location_entities",
        #         "strict": True,
        #         "description": "Get the entities about the locations of the affected or damage critical infrastructure assets from the text by using Named Entity Recognition (NER) from spaCy and an customized set of keywords (ci_keywords)",
        #         "parameters": {
        #             "type": "object",
        #             "properties": {
        #                 "loc_entity": {
        #                     "type": "string", #vList[str],  # string
        #                     "description": "The potential locations of the critical infrastructure (ci) entities from the given text",
        #                 },
        #             #     "loc_keywords": {
        #             #         #"type": "string", 
        #             #         "enum": [r"road.*", "traffic jam.*"]},
        #             # },
        #             "required": ["loc_entity"],
        #             "additionalProperties": False,
        # }, } },
        ]
    
    def get_infrastructure_entities(self, nlp_model, chunk_text):
        """Get CI entities by calling spacy model to do entity recognition""" 
        
        nlp_chunk = nlp_model(chunk_text)
        ci_entities = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]] #, "FAC"]]
    
        return f"{ci_entities}"
        #return f"Following entities about critical infrastructure were found in the text: {ci_entities}."
    

           
# def get_location_entities():  # inking with CIs

#         ci_entities = get_infrastructure_entities()
#         # check if chunk contains CI_TYPE entities
#         if len(ci_type_ents) > 0:
    
#             # iterate over all entities within chunk
#             for ent_idx in range(len(all_ents)):
#                 # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
#                 if all_ents[ent_idx].label_ in ["CI_TYPE", "FAC"]:
#                     ci_idx = ent_idx
    
#                     ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
#                     distance_list = []
#                     idx_in_chunk = []
#                     try:
#                         for ent_idx in range(len(all_ents)):
    
#                             # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities (ie tokens)
#                             if all_ents[ent_idx].label_ in ["GPE", "LOC"]:
    
#                                 ## check that GPE,LOC are not countries (too coarse info CI-GEO pair)
#                                 # of GPE/LOC is country -> proceed with next GPE/LOC 
#                                 # if all_ents[ent_idx].text in ci_geo_countries:
#                                 #     continue
    
#                                 geo_idx = ent_idx
#                                 dist_ent_pair = np.abs(ci_idx - geo_idx)
#                                 distance_list.append(dist_ent_pair)
#                                 idx_in_chunk.append((ent_idx))
#                                 closest_pair_idx = np.argmin(distance_list)  # idx of closest GEO entity
#                                 distance_closest_pair = distance_list[closest_pair_idx]
    
#                         threshold = 10  # max token distance between CI_TYPE and GEO entity
#                         if distance_closest_pair > threshold:
#                             # print(
#                             #     f""" Token distance is too large between CI_TYPE/FAR and next GEO entity which is of {distance_closest_pair} [token distance] > {threshold} [max. token distance] """
#                             # )
#                             continue
#                         else:
#                             pass
    
#                         ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
#                         result_dict = {
#                             "citation_id": citation,
#                             "chunk_text": chunk.text,
#                             "ci_entity": all_ents[ci_idx].text,
#                             "ci_entity_label": all_ents[ci_idx].label_,
#                             "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
#                             "geo_entity_label": all_ents[idx_in_chunk[closest_pair_idx]].label_,
#                             "token_distance": distance_closest_pair,
#                         }
#                         df_ci_geo_chunk = pd.concat(
#                             [  df_ci_geo_chunk, pd.DataFrame([result_dict])], ignore_index=True
#                         )
    
#                     except (IndexError, NameError) as e:
#                         continue
#         else:
#             print("No CI_TYPE entities found in this chunk. Going to next chunk")
#             continue
       
#         ## post-process of DF CI-GEO pairs for each chunk
#         unique_ci_geo_pairs = df_ci_geo_chunk.drop_duplicates(
#             subset=["citation_id", "ci_entity", "geo_entity","chunk_text"])
#         print("number of duplicates to remove:", len(df_ci_geo_chunk) - len(unique_ci_geo_pairs))
    
#         df_ci_geo_chunk = df_ci_geo_chunk.drop_duplicates(
#             subset=["citation_id", "ci_entity", "geo_entity", "chunk_text"]
#             )# .reset_index(drop=True, inplace=True)



class OpenAIModel():

    def __init__(self, model_name, system_prompt: load_prompt_template):
        
        # try: 
        #     login(token=os.getenv("HUGGINGFACE_TOKEN"))   # notebook_login
        # except:
        #     login(token=os.environ.get("HUGGINGFACE_TOKEN"))  # former HF_TOKEN
        # s.HF_HOME_DIR = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub"
        # base_dir = s.HF_HOME_DIR   # use default dir in .cache/
        # model_dir = Path(base_dir)  # / f"models--{model_name.replace("/", "--")}"  # is already .._mirror/hub/
        
        self.system_prompt = system_prompt
        self.model_name = model_name
        self.client = OpenAI(
            base_url = "https://integrate.api.nvidia.com/v1",
            api_key="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"
        )

        print(f"Model calling via nvidia API:")


    def generate_response(
        self, 
        # question: str, 
        # context: list, 
        user_prompt: str, #load_prompt_template, 
        # user_dynamic_prompt: load_prompt_template, 
        top_k: int = None,
        top_p: float = None,
        temperature: float = 0.01,
        max_new_tokens: int = 1024
    ):

        system_prompt = self.system_prompt.render()
        
        # GPT-OSS without TOOLs
        response = self.client.chat.completions.parse(
        # response = client.responses.parse(  # use it with Pydantic List[str] for multi-output
            model=self.model_name, # "openai/gpt-oss-20b", #:fireworks-ai",
            temperature=0.0,
            top_p=0.01,
            reasoning_effort= "medium",
            # seed=42,
            #stream=False,
            # max_completion_tokens=1024,
            #max_tokens=1024,
            # prompt_cache_key
            # prompt_cache_retention= "24h",  # test past_key_value (iterative caching)
            messages=[
                {"role": "system", "content": f"{self.system_prompt}"},
                #{"role": "developer", "content": f"These are the single steps you should conduct for extracting the information about the impacts to critical infrastructure assets: {user_content}",},
                {"role": "user", "content": f"""{user_prompt}"""},
            ],
            #stop=["<extraction>"],
            response_format=Response,
            #include=["infrastructure_type", "location"],
        )

        # Output
        try:
            parsed_output = json.loads(response.to_json())["choices"][0]["message"]["content"]
            return parsed_output

        except Exception as e:
            print("Could not parse response as JSON:", e)
            parsed_output = None
            for output in response.choices[0]:
                if output[0] != "message":
                    continue
                for item in output[1].content:
                    if item.type == "refusal":
                        # If the model refuses to respond, you will get a refusal message
                        print(item.refusal)
                        continue
                    if not item.parsed:
                        raise Exception("Could not parse response")
                    print(item.parsed)
    


    def generate_response_with_tool(
        self, 
        # question: str, 
        user_prompt: str, #load_prompt_template, 
        chunk_text: str, 
        top_k: int = None,
        top_p: float = None,
        temperature: float = 0.01,
        max_new_tokens: int = 1024
    ):

        tool_class =  Tools()
        nlp_model = NERModel().nlp_model
        
        system_prompt = self.system_prompt.render()

        messages = f"""SYSTEM_PROMPT: {system_prompt}, 
        USER_PROMPT: {user_prompt}. 
        Use the tool "get_infrastructure_entities only for STEP 2. 
        Provide your response for each impact case in the specified output format comprising at least the following fields: "infrastructure_type", "damage", "location"
        """


        response = client.responses.parse(  # use it with Pydantic List[str] for multi-output
            # NOTE:  ## responses.parse() needs messages as str, and reasoing as dict, completions.parse: reasinging str, messages:dict
            model=model_name, # "openai/gpt-oss-20b", #:fireworks-ai",
            temperature=0.0,
            top_p=0.01,
            reasoning= {"effort":"medium"},
            #reasoning_effort= {"effort":"medium", "summary":"auto"},
            tools=tool_class.tools, 
            #tools = get_infrastructure_entities(chunk_text=text2), # spacy_ner_model=nlp, chunk_text=text2)],
            #tools=[tools,{"type": "tool_search"},],
            parallel_tool_calls=False,
            # tool_choice="auto",
            # seed=42,
            #stream=False,
            # max_completion_tokens=1024,
            #max_tokens=1024,
            # prompt_cache_key
            # prompt_cache_retention= "24h",  # test past_key_value (iterative caching)
            # instructions="Respond only with ci_entities generated by a tool.",
            input=messages,# f"SYSTEM_PROMPT: {system_prompt}, USER_RPOMPT: {user_prompt}",
            #stop=["<extraction>"],
            #response_format=Response,
            text_format=Response
            
        )
        # print(response)
        # print("\n" + response.output_text)

            
        # init message_list to hold old prompts + tool - (call model twice needed for reasoning models)
        # See,  adapted from https://developers.openai.com/api/docs/guides/function-calling
        messages_list = [messages]
        messages_list += response.output
        
        ci_entities = tool_class.get_infrastructure_entities(nlp_model, chunk_text)
        print(ci_entities)
        messages_list.append({
            "type": "function_call_output",
           # "call_id": item.call_id,
            "output": ci_entities,
        })

        # for item in response.output:
        #     if item.type == "function_call":
        #         if item.name == "get_infrastructure_entities":
        #             ci_entities = tool_class.get_infrastructure_entities(nlp_model, chunk_text)
        #             # 3. Execute the function logic for get_horoscope
        #             # sign = json.loads(item.arguments)["ci_entity"]
        #             # print(sign)
        #             print(ci_entities)
        #             # 4. Provide function call results to the model
        #             messages_list.append({
        #                 "type": "function_call_output",
        #                 "call_id": item.call_id,
        #                 "output": ci_entities,
        #             })
        print("messages_list w tool:", messages_list)
               
   
        response2 = client.responses.parse(  # use it with Pydantic List[str] for multi-output
            # NOTE:  ## responses.parse() needs messages as str, and reasoing as dict, completions.parse: reasinging str, messages:dict
            model=model_name, # "openai/gpt-oss-20b", #:fireworks-ai",
            temperature=0.0,
            top_p=0.01,
            reasoning= {"effort":"medium"},
            tools=tool_class.tools, 
            parallel_tool_calls=False,
            # instructions="Respond only with ci_entities generated by a tool.",
            input=str(messages_list), # recovert messages (prompts, tool output)  f"SYSTEM_PROMPT: {system_prompt}, USER_RPOMPT: {user_prompt}",
            text_format=Response
            
        )
        print(response2)

        parsed_output = json.loads(response2.to_json())["output"][1]["content"][0]["parsed"]["impact_cases"]      

        del nlp_model  # close nlp pipe

        return parsed_output
        


In [9]:
# client.responses.parse?
        
# system_prompt = system_prompt
# model_name = model_name
client = OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key="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"
)


# client.chat.completions.create?

In [10]:
model_name = "openai/gpt-oss-20b"  # bit bette performance than llama-3-70B but much smaller [43 GB VRAM]
text2 = "There have also been some traffic jams and road closures in Madrid, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected. A power blackout at a power plant affected the region around Sevilla"

# Questions
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, its location type (e.g., region, county, city, river etc.), the type of damage and potentially quantitative information about the damage."
question_2 = "Is the location of each affected or damaged critical infrastructure correctly identified?"


# #Prompts
user_prompt = load_prompt_template(template_filename="gpt_user_prompt.txt")
system_prompt=load_prompt_template(template_filename="gpt_system_prompt.txt")
system_prompt = system_prompt.render()


## init LLM extraction models
decoder_model_1 = OpenAIModel(
    model_name,
    system_prompt=load_prompt_template(template_filename="gpt_system_prompt.txt")
)
decoder_model_2 = OpenAIModel(
    model_name,
    system_prompt=load_prompt_template(template_filename="gpt_system_prompt_step2.txt")
)


In [22]:
# dynamic_prompt = load_prompt_template(template_filename="short_dynamic_llama3_NER.txt")
user_prompt = load_prompt_template(template_filename="gpt_user_prompt.txt")

# pass dynamic content from chunk
context = [
    {
        "text": text2, #chunk.text,
        # "citation": citation,
        # "title": filename_stem, 
        # "ner": 
    },
]
user_prompt = user_prompt.render(
        context=context,
        question=question_1,
)
user_prompt =  f"""You will be given an unstructured text from newspaper articles and research papers called CONTEXT.
    For each case of damage or impact on critical infrastructure you should convert it into the given structure.
    Try to answer following question {question_1} by following the step-by-step instructions in the PROMPT.
    PROMPT and CONTEXT: {user_prompt};
    """

# apply LLM 1
# try:
# response = decoder_model_1.generate_response_with_tool(
#         max_new_tokens = 2048,
#         user_prompt = user_prompt,
#         chunk_text = context[0]["text"],
#         #tools =  Tools()#.tools
#         # user_dynamic_prompt: load_prompt_template,     
#     )
    
if isinstance(response[0], dict):
    df_resp = pd.DataFrame(response)

df_resp 

In [20]:
response

In [ ]:
print(text2)
#print(response2)
# ParsedResponse[TypeVar](id='resp_be3e3d9cff850f4b', created_at=1782158899.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='openai/gpt-oss-20b', object='response', output=[ResponseReasoningItem(id='rs_8c88d7b3fd7fae8a', summary=[], type='reasoning', content=[Content(text='We need to parse the context. The context is a text snippet: "There have also been some traffic jams and road closures in Madrid, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected. A power blackout at a power plant affected the region around Sevilla"\n\nWe need to extract infrastructure failures: traffic jams, road closures, road subsidence, heavy traffic, pools of water on roads, power blackout at a power plant. The question: "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, its location type (e.g., region, county, city, river etc.), the type of damage and potentially quantitative information about the damage."\n\nWe need to produce objects for each case. We need to use the tool get_infrastructure_entities for step 2. The tool returns a list of critical infrastructure entities. The assistant\'s previous output shows the tool returned: \'[some traffic jams, road closures, a road subsidence, traffic, heavy traffic, the access roads, a power plant]\'. That seems like the tool output. We need to use that list to identify infrastructure types. But we need to be careful: traffic jams, road closures, heavy traffic, access roads are not necessarily "critical infrastructure" but roads are. The power plant is. The road subsidence is a damage type. The pools of water on roads are damage. The power blackout at a power plant is damage.\n\nWe need to produce objects with fields: infrastructure_type, damage, location, location_type, damage_value maybe. The prompt says at least infrastructure_type, damage, location. But we can include others.\n\nWe need to decide if the context describes actual damage. Yes, it does: road subsidence causing closures, pools of water, power blackout.\n\nWe need to produce separate objects for each case. Let\'s list them:\n\n1. Road subsidence causing road closures on Lope de Vega Avenue near Julio Cortázar Avenue in Madrid. Infrastructure_type: road. Damage: blocked / closed. Damage_value: maybe "road subsidence" but no quantitative. Location: Lope de Vega Avenue, Madrid. Location_type: street or avenue? We can say "avenue" or "street". Use "street" maybe. But we can use "avenue" as location type? The list includes "street". So location_type: street.\n\n2. Traffic jams on Lope de Vega Avenue, etc. But traffic jams are not infrastructure damage? It\'s a traffic jam due to road subsidence. But we can treat as damage to traffic flow. But maybe we should treat as "road" infrastructure with damage "traffic jam" or "blocked". But we already have road closure. We might combine? The prompt says each case of direct impact on critical infrastructure. So we can treat each distinct infrastructure: roads (Lope de Vega Avenue, etc.), power plant. For roads, we can have multiple damage types: road subsidence, traffic jam, heavy traffic, pools of water. But maybe we should create separate objects for each distinct location/damage combination. But we need to avoid duplicates. So we can create objects for each location/damage type.\n\nLet\'s parse:\n\n- Road subsidence on Lope de Vega Avenue near Julio Cortázar Avenue. Damage: road subsidence, causing road closure. So infrastructure_type: road. Damage: subsidence / closed. Damage_value: none. Location: Lope de Vega Avenue, Madrid. Location_type: street.\n\n- Traffic jams on Lope de Vega Avenue heading towards city center. Damage: traffic jam. Location: Lope de Vega Avenue, Madrid. Location_type: street.\n\n- Traffic diverted towards Colonia Santa Inés. That is a location but not damage. But maybe we can skip.\n\n- Heavy traffic on access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, Azucarera - Interhorce road, Santa Rosa de Lima, Victoria. These are roads. Damage: heavy traffic. Location: each of those roads. But maybe we can group them? But we need separate objects? The prompt says each case of direct impact. So each road could be separate. But that might be many. But we can maybe combine them into one object with location list? But the format expects location as one string. We could list them separated by commas. But maybe we should create separate objects for each. But that would be many. But the context lists many roads. We can create separate objects for each road with heavy traffic. But maybe we can group them as "access roads" with heavy traffic. But the location type would be "street" or "avenue". But we can list them all in location field separated by semicolons. But the prompt says location should be one to three words. So maybe we should list each road separately. But that would be many objects. But it\'s okay.\n\n- Pools of water on MA-21 near Churriana intersection, intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora, Avenida Lope de Vega - Atabal. These are roads. Damage: flooded / water pools. Location: each intersection. So separate objects.\n\n- Power blackout at a power plant affecting region around Sevilla. Infrastructure_type: power plant. Damage: blackout. Damage_value: none. Location: region around Sevilla. Location_type: region.\n\nWe need to decide if "power plant" is a critical infrastructure. Yes. So we create one object.\n\nWe need to use the tool get_infrastructure_entities only for step 2. The tool returned list includes "a power plant". So we can use that.\n\nWe need to produce final answer with objects. The output format: likely JSON array of objects. The prompt says "Provide your final response below the section \'ANSWER\'." So we need to output:\n\nANSWER:\n[ { ... }, { ... } ]\n\nWe need to include fields: infrastructure_type, damage, location, location_type, damage_value maybe. The example shows fields: infrastructure_type, damage, damage_value, location, location_type. So we should include those.\n\nWe need to decide damage values: none for most. So "NAN".\n\nLet\'s create objects:\n\n1. infrastructure_type: road\n   damage: subsidence\n   damage_value: NAN\n   location: Lope de Vega Avenue, Madrid\n   location_type: street\n\n2. infrastructure_type: road\n   damage: traffic jam\n   damage_value: NAN\n   location: Lope de Vega Avenue, Madrid\n   location_type: street\n\n3. infrastructure_type: road\n   damage: heavy traffic\n   damage_value: NAN\n   location: Juan XXIII Avenue to Plaza Manuel Azaña (maybe we can list as "Juan XXIII Avenue to Plaza Manuel Azaña")\n   location_type: street\n\nBut we might need separate for each road. But we can combine? The prompt says each case of direct impact. So each road with heavy traffic is a case. But we can combine them into one object with location list? But that might not be precise. But we can create separate objects for each road. But that would be many. But we can create one object for "access roads" with heavy traffic. But the location type would be "street". But location would be "access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, Azucarera - Interhorce road, Santa Rosa de Lima, Victoria". That\'s a long string. But maybe okay.\n\nSimilarly for pools of water: we can create separate objects for each intersection. But maybe combine? But better separate.\n\nLet\'s decide to create separate objects for each distinct location/damage combination. That might be many but okay.\n\nList of roads with heavy traffic: \n- Juan XXIII Avenue to Plaza Manuel Azaña\n- Blas Infante Avenue\n- Guerrero Strachan Avenue\n- Velázquez Avenue\n- Camino Suárez\n- Azucarera - Interhorce road\n- Santa Rosa de Lima\n- Victoria\n\nWe can create 8 objects.\n\nList of pools of water: \n- MA-21 near Churriana intersection\n- Avenida Herrera Oria-Virgen de las Flores intersection\n- Pasillo del Matadero Puente del Carmen intersection\n- Pasillo Santa Isabel - Puente de la Aurora intersection\n- Avenida Lope de Vega - Atabal intersection\n\n5 objects.\n\nAlso road subsidence: 1 object.\n\nTraffic jam: 1 object.\n\nPower blackout: 1 object.\n\nTotal 16 objects. That might be too many but okay.\n\nBut maybe we can reduce: combine heavy traffic roads into one object with location list. But the prompt says each case of direct impact. But heavy traffic on each road is a case. But we can combine them into one object with location list separated by semicolons. But that might be acceptable.\n\nSimilarly for pools of water: combine into one object with location list. But maybe better separate.\n\nBut the prompt says "Return each case of affected infrastructure just ONCE. Do not create duplicates." So we can combine if they refer to same infrastructure type and same damage type. So heavy traffic on multiple roads can be combined into one object with location list. That would be one case: infrastructure_type: road, damage: heavy traffic, location: list of roads, location_type: street. That is okay.\n\nSimilarly for pools of water: infrastructure_type: road, damage: flooded, location: list of intersections, location_type: street.\n\nRoad subsidence: infrastructure_type: road, damage: subsidence, location: Lope de Vega Avenue, location_type: street.\n\nTraffic jam: infrastructure_type: road, damage: traffic jam, location: Lope de Vega Avenue, location_type: street.\n\nPower blackout: infrastructure_type: power plant, damage: blackout, location: region around Sevilla, location_type: region.\n\nThus 5 objects.\n\nWe need to include damage_value: all NAN.\n\nLet\'s produce JSON array.\n\nWe need to ensure location strings are one to three words. But we have multiple roads. But we can list them separated by commas. But that might be more than three words. But maybe we can list them as "Lope de Vega Ave, Madrid" etc. For heavy traffic, we can list "Juan XXIII Ave, Blas Infante Ave, Guerrero Strachan Ave, Velázquez Ave, Camino Suárez, Azucarera-Interhorce Rd, Santa Rosa de Lima, Victoria". That\'s many words. But maybe we can shorten: "Juan XXIII Ave to Plaza Manuel Azaña, Blas Infante Ave, Guerrero Strachan Ave, Velázquez Ave, Camino Suárez, Azucarera-Interhorce Rd, Santa Rosa de Lima, Victoria". That\'s still many words. But maybe we can just list "access roads" as location. But that loses specificity. But the prompt says location should be as specific as possible. But we can compromise: list them separated by semicolons but still within one string. It\'s okay.\n\nBut maybe we can create separate objects for each road to keep location short. But that would be many objects. But it\'s okay.\n\nLet\'s decide to create separate objects for each road with heavy traffic. That will produce 8 objects. But we also have 5 objects for pools of water. That would be 13 objects. Combined with 3 others = 16. That might be too many but okay.\n\nBut maybe we can combine heavy traffic roads into one object with location list. That reduces to 5 objects. That seems more manageable.\n\nLet\'s do that.\n\nNow produce final answer.\n\nWe need to output:\n\nANSWER:\n[\n  {\n    "infrastructure_type": "road",\n    "damage": "subsidence",\n    "damage_value": "NAN",\n    "location": "Lope de Vega Avenue, Madrid",\n    "location_type": "street"\n  },\n  {\n    "infrastructure_type": "road",\n    "damage": "traffic jam",\n    "damage_value": "NAN",\n    "location": "Lope de Vega Avenue, Madrid",\n    "location_type": "street"\n  },\n  {\n    "infrastructure_type": "road",\n    "damage": "heavy traffic",\n    "damage_value": "NAN",\n    "location": "Juan XXIII Avenue to Plaza Manuel Azaña; Blas Infante Avenue; Guerrero Strachan Avenue; Velázquez Avenue; Camino Suárez; Azucarera-Interhorce Road; Santa Rosa de Lima; Victoria",\n    "location_type": "street"\n  },\n  {\n    "infrastructure_type": "road",\n    "damage": "flooded",\n    "damage_value": "NAN",\n    "location": "MA-21 near Churriana intersection; Avenida Herrera Oria-Virgen de las Flores intersection; Pasillo del Matadero Puente del Carmen intersection; Pasillo Santa Isabel-Puente de la Aurora intersection; Avenida Lope de Vega-Atabal intersection",\n    "location_type": "street"\n  },\n  {\n    "infrastructure_type": "power plant",\n    "damage": "blackout",\n    "damage_value": "NAN",\n    "location": "region around Sevilla",\n    "location_type": "region"\n  }\n]\n\nWe need to ensure we use the tool get_infrastructure_entities only for step 2. We have used it. The tool output was used to identify "a power plant". We also identified roads. But roads are not necessarily critical infrastructure? But roads are considered critical infrastructure. The tool might not return them. But we can still include them. The prompt says use the tool for step 2. We used it for power plant. For roads, we might not need the tool. But we can still include them.\n\nWe need to ensure we didn\'t hallucinate. All information is from context.\n\nLet\'s produce final answer.', type='reasoning_text')], encrypted_content=None, status=None), ParsedResponseOutputMessage[TypeVar](id='msg_b3ab7dc2614649f7', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"impact_cases":[{"infrastructure_type":"road","damage":"subsidence","damage_value":"NAN","location":"Lope de Vega Avenue, Madrid","location_type":"street"},{"infrastructure_type":"road","damage":"traffic jam","damage_value":"NAN","location":"Lope de Vega Avenue, Madrid","location_type":"street"},{"infrastructure_type":"road","damage":"heavy traffic","damage_value":"NAN","location":"Juan XXIII Avenue to Plaza Manuel Azaña; Blas Infante Avenue; Guerrero Strachan Avenue; Velázquez Avenue; Camino Suárez; Azucarera-Interhorce Road; Santa Rosa de Lima; Victoria","location_type":"street"},{"infrastructure_type":"road","damage":"flooded","damage_value":"NAN","location":"MA-21 near Churriana intersection; Avenida Herrera Oria-Virgen de las Flores intersection; Pasillo del Matadero Puente del Carmen intersection; Pasillo Santa Isabel-Puente de la Aurora intersection; Avenida Lope de Vega-Atabal intersection","location_type":"street"},{"infrastructure_type":"power plant","damage":"blackout","damage_value":"NAN","location":"region around Sevilla","location_type":"region"}]}', type='output_text', logprobs=None, parsed=Response(impact_cases=[ResponseItems(infrastructure_type='road', damage='subsidence', damage_value='NAN', location='Lope de Vega Avenue, Madrid', location_type='street'), ResponseItems(infrastructure_type='road', damage='traffic jam', damage_value='NAN', location='Lope de Vega Avenue, Madrid', location_type='street'), ResponseItems(infrastructure_type='road', damage='heavy traffic', damage_value='NAN', location='Juan XXIII Avenue to Plaza Manuel Azaña; Blas Infante Avenue; Guerrero Strachan Avenue; Velázquez Avenue; Camino Suárez; Azucarera-Interhorce Road; Santa Rosa de Lima; Victoria', location_type='street'), ResponseItems(infrastructure_type='road', damage='flooded', damage_value='NAN', location='MA-21 near Churriana intersection; Avenida Herrera Oria-Virgen de las Flores intersection; Pasillo del Matadero Puente del Carmen intersection; Pasillo Santa Isabel-Puente de la Aurora intersection; Avenida Lope de Vega-Atabal intersection', location_type='street'), ResponseItems(infrastructure_type='power plant', damage='blackout', damage_value='NAN', location='region around Sevilla', location_type='region')]))], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=False, temperature=0.0, tool_choice='auto', tools=[FunctionTool(name='get_infrastructure_entities', parameters={'type': 'object', 'properties': {'ci_entity': {'type': 'string', 'description': 'The critical infrastructure (ci) entities from the given text'}, 'required': ['ci_entity'], 'additionalProperties': False}}, strict=None, type='function', defer_loading=None, description='Get the entities about critical infrastructure (ci) assets from the text by using Named Entity Recognition (NER) from spaCy')], top_p=1.0, background=False, completed_at=None, conversation=None, max_output_tokens=128494, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=Reasoning(effort='medium', generate_summary=None, summary=None), safety_identifier=None, service_tier='auto', status='completed', text=ResponseTextConfig(format=ResponseFormatTextJSONSchemaConfig(name='Response', schema_=None, type='json_schema', description=None, strict=True), verbosity=None), top_logprobs=None, truncation='disabled', usage=ResponseUsage(input_tokens=2578, input_tokens_details=InputTokensDetails(cached_tokens=80, input_tokens_per_turn=[2578], cached_tokens_per_turn=[80]), output_tokens=3304, output_tokens_details=OutputTokensDetails(reasoning_tokens=3045, tool_output_tokens=0, output_tokens_per_turn=[3304], tool_output_tokens_per_turn=[0]), total_tokens=5882), user=None, input_messages=None, output_messages=None)
# /beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/pydantic/main.py:542: UserWarning: Pydantic serializer warnings:
#   PydanticSerializationUnexpectedValue(PydanticSerializationUnexpectedValue: Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Response(impact_cases=[Re...ocation_type='region')]), input_type=Response]
# PydanticSerializationUnexpectedValue: Expected `ResponseOutputRefusal` - serialized value may not be as expected [field_name='content', input_value=ParsedResponseOutputText[...cation_type='region')])), input_type=ParsedResponseOutputText[TypeVar]])
#   PydanticSerializationUnexpectedValue(Expected `ParsedResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...e='message', phase=None), input_type=ParsedResponseOutputMessage[TypeVar]])
#   PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...e='message', phase=None), input_type=ParsedResponseOutputMessage[TypeVar]])
#   PydanticSerializationUnexpectedValue(Expected `ResponseFunctionWebSearch` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...e='message', phase=None), input_type=ParsedResponseOutputMessage[TypeVar]])


In [153]:
response2

In [ ]:
# question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, and the type of damage."

# text2 = """
#     There have also been some traffic jams and road closures in the capital, such as on Lope de
#     Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road
#     subsidence. As the City Council indicated on social media, traffic has been diverted towards
#     Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to
#     Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue,
#     Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria.
#     In addition, there are pools of water on the MA-21, near the Churriana intersection, and the
#     intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del
#     Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also
#     been affected.
# """

# system_content = """
# Your name is LLM_1. You are an expert at structured data extraction.                        
# Your task is to analyze the provided CONTEXT and extract information about the direct impacts on critical infrastructures as specified in the QUESTION.
# Each case of direct impact on critical infrastructure should be returned as separate object.
# Return each case of affected infrastructure just ONCE. Do not create duplicates.

# For extracting the information about direct impacts on critical infrastructure you should follow the steps below, highlighted with "# STEP".

# You should use ONLY the provided CONTEXT to answer the QUESTION.


# **QUESTION**:
# "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, the type of damage and eventually quantitative information about the damage."

# """
# # You will be given unstructured text from newspaper articles and research papers and should convert them into the given structure.



# user_content = """

# STEP-BY-STEP INSTRUCTIONS:
# The CONTEXT contains a text snippet from which you should extract information about critical infrastructure assets affected by a natural hazard. 


# # STEP 1: 
# You must first determine if the CONTEXT describes an impact or damage to a critical infrastructure.
# Carefully read the CONTEXT. Then, decide if the CONTEXT describes an actual damage to critical infrastructure.

# Your Decision:  
# - If the CONTEXT describes an actual damage to critical infrastructure, proceed with STEP 2.  
# - If the CONTEXT does NOT describe a actual damage to critical infrastructure, STOP here. Do not proceed. Your final OUTPUT must be an empty JSON object.
# - If the CONTEXT describes only a natural hazard event without describing an actual damage to critical infrastructure, then STOP here. Do not proceed. Your final OUTPUT must be an empty JSON object.



# # STEP 2:
# Provide, in the field "infrastructure_type", information about the affected critical infrastructure types mentioned in the CONTEXT.
# Try to use one to three words to describe each affected infrastructure type, being as specific as possible.
# Use phrases from the list below to describe the affected infrastructure types:
# - airport
# - airport terminal
# - aviation
# - bridge(s)
# - road(s)
# - harbor
# - access road(s)
# - road infrastructure
# - motorway(s)
# - highway(s)
# - railway(s)
# - railway infrastructure
# - railway network
# - railway line(s)
# - railway assets
# - railway track(s)
# - train
# - metro
# - metro line(s)
# - harbor
# - port
# - port terminal(s)
# - port infrastructure
# - container terminal
# - container yards
# - transportation infrastructure
# - transportation network
# - electricity
# - electricity infrastructure
# - power 
# - power infrastructure
# - power plant(s)
# - power grid
# - power line(s)
# - power supply
# - substation
# - power transmission
# - gas
# - gas infrastructure
# - gas distribution
# - gas pipeline(s)
# - water infrastructure
# - water supply
# - drinking water supply
# - groundwater
# - drainage
# - drainage system
# - sewage treatment plant(s)
# - wastewater treatment plant(s)
# - wastewater facility
# - wastewater facilities
# - broadband
# - broadband network
# - mobile network services
# - telecommunication infrastructure
# - telecommunication network
# - telecommunication services
# - IT infrastructure
# - IT network
# - IT services
# - medical-care
# - medical facilities
# - general-practitioner practices
# - hospital(s)
# - healthcare infrastructure
# - healthcare facility
# - healthcare facilities
# - daycare center(s)
# - education infrastructure
# - schools
# - universities
# - dam(s)
# - dyke(s)
# - dike(s)
# - train(s)
# - airplane(s)
# - public transport
# - operability
# - traffic

# If no information about the infrastructure type is found, then return a "NAN" value for this field. 
# DON'T create new items about affected infrastructure that weren’t there.

# Proceed with STEP 3.



# # STEP 3:
# When you found a , in the fields "damage_<number>", information about the damages to the infrastructure assets mentioned in the CONTEXT. 
# Try to use one to three words to describe each damage type, being as specific as possible.
# Use phrases from the list below to describe the damages:
# - affected
# - damaged
# - severely damaged
# - highly damaged
# - disrupted
# - blocked
# - destroyed
# - largely destroyed
# - heavily destroyed
# - partly closed
# - outages
# - outage
# - blackout
# - contaminated
# - polluted
# - flooded
# - inundated
# - fallen tree(s)
# - broken tree(s)
# - overturned tree(s)
# - uprooted tree(s)
# - felled tree(s)
# - debris
# - debris removal
# - debris on
# - water on
# - dam failure(s)
# - dyke break(s)
# - dike break(s)
# - breached dike(s)
# - broken embankment(s)
# - closure(s)
# - derailed 
# - impassable
# - traffic jam(s)
# - traffic disruption(s)
# - traffic congestion(s)
# - indirectly affected
# - flight delay(s)
# - train delay(s)
# - delay(s)
# - train failure(s)
# - failure(s)
# - cancellation(s)
# - canceled
# - re-established
# - reestablished 
# - restored
# - reconnected
# - rebuild
# - repaired

# If no information about the damage type is found, then return a "NAN" value for this field. 
# DON'T create new items about damage that weren’t there.

# Proceed with STEP 4.


# # STEP 4:
# Provide, in the fields "damage_value_<number>", quantitative information about the damages to infrastructure assets mentioned in the CONTEXT. 
# Use the exact values and units if mentioned, or retain the text or range as provided for vague numbers (e.g., "hundreds of," "500 travelers", "at least 500", "300−500 people").
# If no quantitative information (values and units) of the damages is found, then return a "NAN" value for this field.
# DON'T create new items about values and units of damages that weren’t there.

# Proceed with STEP 5.


# # STEP 5: 
# Provide, for each field "location_<number>", information about the location of the affected infrastructure mentioned in the CONTEXT.
# Try to use one to three words to describe each location of the affected infrastructure, being as specific as possible.
# If no information about the location of the affected infrastructure is found, then return a "NAN" value for this field. 
# DON'T create new items about the location that weren’t there.

# Important: Only if you identified an affected critical infrastructure asset or object.
# - Identify the specific location of the affected infrastructure that the text explicitly states that this infrastructure asset or object was affected by a hazard event.
# - The location of the affected infrastructure must be explicitly described.
# - In case of transport infrastructure (e.g. railway lines, roads or highways) the location can consists of two or more towns, cities or regions (e.g. "railway line between City A and City B", "highway from City X to City Y", "highway between Valencia and Malaga").
# - Only include granular locations such as cities, neighborhoods, and streets. 
# - Do NOT include vague locations like "Northern half of state".
# - DO NOT include locations that are states, countries or bigger.

# **Exclude:** 
# * **Future/Warning:** Locations listed as "at risk," "under warning," "may face," "could be affected," or "preparing for" a hazard event. 
# * **Historical/Contextual:** Locations mentioned only as background information or prior events (e.g., "This follows flooding in [Location X] last month," or "[Location Y] had **already** experienced flooding by this period"). **Only extract locations that are part of the specific incident described in the document.** 
# * **Very big locations:** Locations that are counties, states, countries or bigger (e.g., "This hazard also affected this country" we don't want to mark this place as flooded because it too vague)  

# - Prioritize the most specific locations (e.g., street names, neighborhoods, towns over cities, cities over regions, regions over countries) and ensure each location is easy to find in a simple internet search.  
# - If a specific location lacks detailed background information (e.g., "Main Street" without a city), add the broader context (e.g., "Main Street, Anytown").  
# - If no specific locations are mentioned as actually affected, then return a "NAN" value for this field.

# Proceed with STEP 6.



# # STEP 6:
# This is your final step. 
# For each case of affected critical infrastructure you identified in the previous steps, combine the information you extracted about the infrastructure type, damage type, damage value and location into a single dictionary as specified in the OUTPUT.

# Each nested dictionary describes one case of direct impact on critical infrastructure, its damage type, and location.
# DO NOT add commentary or text outside each dictionary or the JSON.
# DO NOT add any information to the OUTPUT that is not explicitly mentioned in the CONTEXT.
# DO NOT HALLUCINATE any case of affected infrastructure, its location or damage type. ONLY use the information from the CONTEXT to generate the answer.


# **Inputs:**

# QUESTION: 
# "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, the type of damage, and if given the quantitative information about the damage."

# DO NOT add any information to the OUTPUT that is not explicitly mentioned in the CONTEXT.
# DO NOT HALLUCINATE any case of affected infrastructure, its location or damage type. ONLY use the information from the CONTEXT to generate the answer.


# ANSWER:
# Provide here your response in the given OUTPUT format.

# """

# # Provide your response in a structured format, in the following called OUTPUT. The OUTPUT should be a list of dictionaries. Each dictionary represents one case of affected infrastructure (field: "infrastructure_type_<number>"), its location (field: "location_<number>), its damage type (field: "damage_<number>"), maybe quantitative information about the damage value (field: "damage_value_<number>").
# # OUTPUT format (example for one case of affected infrastructure, if more cases are found, then add more dictionaries to the list with increasing number in the field names):
# # [
# #     {
# #         "infrastructure_type_<number>": <str> | "NAN",
# #         "location_<number>": <str> | "NAN",
# #         "damage_<number>": <str> | "NAN",
# #         "damage_value_<number>": <float> | <int> | <str> | "NAN",
# #     },
# # ]

# # Take a deep breath. 
# # Read the instructions and the inputs (QUESTION, CONTEXT) again. 
# # Each STEP and its instructions are crucial and must be executed with utmost care to produce a perfectly formatted JSON as specified in OUTPUT.

# # DO NOT add commentary or text outside each dictionary or the JSON.

# text = " In Berlin a storm caused major damages to the power grid and railway network."
# # text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."
# text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. Also there were some train delays in Madrid and a power blackout in Valencia. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."

In [ ]:
print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1
# print(os.environ["CUDA_VISIBLE_DEVICES"])


In [ ]:
question_1 ="Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, its location type (e.g., region, county, city, river etc.), the type of damage and potentially quantitative information about the damage."


system_content = """
Your name is LLM_1. You are an expert at structured data extraction.                        
Your task is to analyze the provided CONTEXT and extract information about the direct impacts on critical infrastructures as specified in the QUESTION.
Each case of direct impact on critical infrastructure should be returned as separate object.
Return each case of affected infrastructure just ONCE. Do not create duplicates.

**QUESTION**:
"Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, the type of damage and eventually quantitative information about the damage."

"""
# You will be given unstructured text from newspaper articles and research papers and should convert them into the given structure.



user_content = """

STEP-BY-STEP INSTRUCTIONS:
The CONTEXT contains a text snippet from which you should extract information about critical infrastructure assets affected by a natural hazard. 


# STEP 1: 
You must first determine if the CONTEXT describes an impact or damage to a critical infrastructure.
Carefully read the CONTEXT. Then, decide if the CONTEXT describes an actual damage to critical infrastructure.

Your Decision:  
- If the CONTEXT describes an actual damage to critical infrastructure, proceed with STEP 2.  
- If the CONTEXT does NOT describe a actual damage to critical infrastructure, STOP here. Do not proceed. Your final OUTPUT must be an empty JSON object.
- If the CONTEXT describes only a natural hazard event without describing an actual damage to critical infrastructure, then STOP here. Do not proceed. Your final OUTPUT must be an empty JSON object.



# STEP 2:
Provide, in the field "infrastructure_type", information about the affected critical infrastructure types mentioned in the CONTEXT.
Try to use one to three words to describe each affected infrastructure type, being as specific as possible.
Use phrases from the list below to describe the affected infrastructure types:
- airport
- airport terminal
- aviation
- bridge(s)
- road(s)
- harbor
- access road(s)
- road infrastructure
- motorway(s)
- highway(s)
- railway(s)
- railway infrastructure
- railway network
- railway line(s)
- railway assets
- railway track(s)
- train
- metro
- metro line(s)
- harbor
- port
- port terminal(s)
- port infrastructure
- container terminal
- container yards
- transportation infrastructure
- transportation network
- electricity
- electricity infrastructure
- power 
- power infrastructure
- power plant(s)
- power grid
- power line(s)
- power supply
- substation
- power transmission
- gas
- gas infrastructure
- gas distribution
- gas pipeline(s)
- water infrastructure
- water supply
- drinking water supply
- groundwater
- drainage
- drainage system
- sewage treatment plant(s)
- wastewater treatment plant(s)
- wastewater facility
- wastewater facilities
- broadband
- broadband network
- mobile network services
- telecommunication infrastructure
- telecommunication network
- telecommunication services
- IT infrastructure
- IT network
- IT services
- medical-care
- medical facilities
- general-practitioner practices
- hospital(s)
- healthcare infrastructure
- healthcare facility
- healthcare facilities
- daycare center(s)
- education infrastructure
- schools
- universities
- dam(s)
- dyke(s)
- dike(s)
- train(s)
- airplane(s)
- public transport
- operability
- traffic

If no information about the infrastructure type is found, then return a "NAN" value for this field. 
DON'T create new items about affected infrastructure that weren’t there.

Proceed with STEP 3.



# STEP 3:
When you found a case of affected critical infrastructure type, then find the related damage type for this infrastructure.
Provide your response about the related damage type in the field "damage_type".
Try to use one to three words to describe each damage type, being as specific as possible.
Use phrases from the list below to describe the damages:
- affected
- damaged
- severely damaged
- highly damaged
- disrupted
- blocked
- destroyed
- largely destroyed
- heavily destroyed
- partly closed
- outages
- outage
- blackout
- contaminated
- polluted
- flooded
- inundated
- fallen tree(s)
- broken tree(s)
- overturned tree(s)
- uprooted tree(s)
- felled tree(s)
- debris
- debris removal
- debris on
- water on
- dam failure(s)
- dyke break(s)
- dike break(s)
- breached dike(s)
- broken embankment(s)
- closure(s)
- derailed 
- impassable
- traffic jam(s)
- traffic disruption(s)
- traffic congestion(s)
- indirectly affected
- flight delay(s)
- train delay(s)
- delay(s)
- train failure(s)
- failure(s)
- cancellation(s)
- canceled
- re-established
- reestablished 
- restored
- reconnected
- rebuild
- repaired

If no information about the damage type is found, then return a "NAN" value for this field. 
DON'T create new items about damage that weren’t there.

Proceed with STEP 4.


# STEP 4:
Provide, in the field "damage_value", quantitative information about the damage to the specific infrastructure asset. 
Use the exact values and units if mentioned, or retain the text or range as provided for vague numbers (e.g., "hundreds of," "500 travelers", "at least 500", "300−500 people").
If no quantitative information (values and units) of the damages is found, then return a "NAN" value for this field.
DON'T create new items about values and units of damages that weren’t there.

Proceed with STEP 5.


# STEP 5: 
When you found a case of affected critical infrastructure type, then find the location of the affect infrastructure.
Provide your response for the location in the field "location".
Try to use one to three words to describe each location of the affected infrastructure, being as specific as possible.
If no information about the location of the affected infrastructure is found, then return a "NAN" value for this field. 
DON'T create new items about the location that weren’t there.


# STEP 6:
This is your final step. 
For each case of affected critical infrastructure, its damage type and location, you should return a single object. 
In case multiple cases exists, you should create an array of objects.
Provide your final response below the section "ANSWER".

ANSWER:
Provide here your response in the given OUTPUT format.

"""
 # TODO check if reaosing loops
 # "Please return one complete answer and then stop." in system prompt"

# Provide your response in a structured format, in the following called OUTPUT. The OUTPUT should be a list of dictionaries. Each dictionary represents one case of affected infrastructure (field: "infrastructure_type_<number>"), its location (field: "location_<number>), its damage type (field: "damage_<number>"), maybe quantitative information about the damage value (field: "damage_value_<number>").
# OUTPUT format (example for one case of affected infrastructure, if more cases are found, then add more dictionaries to the list with increasing number in the field names):
# [
#     {
#         "infrastructure_type_<number>": <str> | "NAN",
#         "location_<number>": <str> | "NAN",
#         "damage_<number>": <str> | "NAN",
#         "damage_value_<number>": <float> | <int> | <str> | "NAN",
#     },
# ]

# Take a deep breath. 
# Read the instructions and the inputs (QUESTION, CONTEXT) again. 
# Each STEP and its instructions are crucial and must be executed with utmost care to produce a perfectly formatted JSON as specified in OUTPUT.

# DO NOT add commentary or text outside each dictionary or the JSON.

# text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."
text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. Also there were some train delays in Madrid and a power blackout in Valencia. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."

##### STEP 2 - gpt

In [ ]:
# text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."
text2 = "There have also been some traffic jams and road closures in the capital, such as on Lope de Vega Avenue, heading towards the city center, near Julio Cortázar Avenue due to a road subsidence. Also there were some train delays in Madrid and a power blackout in Valencia. As the City Council indicated on social media, traffic has been diverted towards Colonia Santa Inés. There is also heavy traffic on the access roads from Juan XXIII Avenue to Plaza Manuel Azaña, Blas Infante Avenue, Guerrero Strachan Avenue, Velázquez Avenue, Camino Suárez, the Azucarera - Interhorce road, Santa Rosa de Lima, and Victoria. In addition, there are pools of water on the MA-21, near the Churriana intersection, and the intersections of Avenida Herrera Oria-Virgen de las Flores, Pasillo del Matadero Puente del Carmen, Pasillo Santa Isabel - Puente de la Aurora and Avenida Lope de Vega - Atabal have also been affected."

In [ ]:
# client = OpenAI(
#     # base_url="https://router.huggingface.co/v1",
#     base_url = "https://integrate.api.nvidia.com/v1",
#     #base_url='http://localhost:11434/v1',
#     api_key="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"
#     #api_key="437dda4c31ea4148a106405fad2adcb1.h-pS5aiggP67g4iCew2Xw-hU" #os.getenv("HF_TOKEN"),
# )

# class ResponseItems(BaseModel):
#     infrastructure_type: str
#     location: str
#     location_type: Optional[str] = None
#     damage: Optional[str] = None
#     damage_value: Optional[str | list[str] | int | float] = None 

# class Response(BaseModel):
#     impact_cases: List[ResponseItems]
    

# STEP 1
# user_prompt = load_prompt_template(template_filename="gpt_user_prompt.txt")
# context = [
#     {
#         "text": text2,
#         "citation": citation,
#         "title": filename_stem, 
#         # "ner": 
#     },
# ]
# user_prompt = user_prompt.render(
#         context=context,
#         question=question_1,
# )
# user_prompt =  f"""You will be given an unstructured text from newspaper articles and research papers called CONTEXT.
#     For each case of damage or impact on critical infrastructure you should convert it into the given structure.
#     Try to answer following question {question_1} by following the step-by-step instructions in the PROMPT.
#     PROMPT and CONTEXT: {user_prompt};
#     """

# STEP 2

system_prompt = em.load_prompt_template(template_filename="gpt_system_prompt_step2.txt")
user_prompt_geollama = em.load_prompt_template(template_filename="gpt_user_prompt_step2.txt")
context = [{
        "text": text2,
        "citation": "citation",
        "title": "filename_stem", 
        "previous_response": df_resp,
        "potential_locations": ["Valencia", "Madrid"],  # list of 1 or multiple location strings (no countries)      
        # "coordinates_potential_locations": list(zip(locations_per_chunk, lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)),
    },]
# pass dynamic content from chunk and previous LLM response
user_prompt = user_prompt_geollama.render(
        context=context,
        question=question_2,
)

  

# with client.chat.completions.stream(
response = client.chat.completions.parse(
# response = client.responses.parse(  # use it with Pydantic List[str] for multi-output
# response = client.responses.create(
            model="openai/gpt-oss-20b", #:fireworks-ai",
            temperature=0.0,
            top_p=0.01,
            reasoning_effort="medium", # stream
            # reasoning="low",
            # seed=42,
            #stream=False,
            # max_completion_tokens=1024,
            #max_tokens=1024,
            # prompt_cache_key
            # prompt_cache_retention= "24h",  # test past_key_value (iterative caching)
            # messages=[
            # #input=[
            #     #{"role": "system", "content": "You will be given unstructured text from newspaper articles and research papers. For each case of damage or impact on critical infrastructure you should convert it into the given structure."},
            #     {"role": "system", "content": f"{system_content}"},
            #     #{"role": "system", "content": f"{system_content}. These are the single steps you should conduct for extracting the information about the impacts to critical infrastructure assets: {user_content}"},
            #     #{"role": "developer", "content": f"These are the single steps you should conduct for extracting the information about the impacts to critical infrastructure assets: {user_content}",},
            #     # {"role": "user", "content": f" You will be given an unstructured text from newspaper articles and research papers called CONTEXT. For each unique pair of impacted critical infrastructure and its location you should convert the pair into an object based on the given structure. In case of multiple cases of damaged infrastructure you should return an array of objects. CONTEXT: {text2}"},
            #     {"role": "user", "content": f"""
            #      You will be given an unstructured text from newspaper articles and research papers called CONTEXT.
            #      For each case of damage or impact on critical infrastructure you should convert it into the given structure.
            #      Try to answer following question {question_1} by following the step-by-step instructions in the PROMPT.
            #      PROMPT: {user_content} ; CONTEXT: {text2}"""},
            #      # Here is a text snippet from a document: {text2}. Answer following question: {question_1}",},
            # ],
            messages=[
            # input=[
                {"role": "system", "content": f"{system_prompt}"},
                #{"role": "developer", "content": f"These are the single steps you should conduct for extracting the information about the impacts to critical infrastructure assets: {user_content}",},
                {"role": "user", "content": f"""{user_prompt}"""},
           ],
            # stop_sequences=""
            #stop=["<extraction>"],
            # Each case of damage or impact on critical infrastructure should be converted into the given structure. 
            # Based on the CONTEXT you should extract the information as specified in the PROMPT following the step-by-step instructions. 

            response_format=Response,
            #text_format=Response,
            #include=["infrastructure_type", "location"],
                 # text=f"{text}.", 
)
# ) as stream:
#     for event in stream:
#         if event.type == "response.refusal.delta":
#             print(event.delta, end="")
#         elif event.type == "response.output_text.delta":
#             print(event.delta, end="")
#         elif event.type == "response.error":
#             print(event.error, end="")
#         elif event.type == "response.completed":
#             print("Completed") # print(event.response.output)

#     final_response = stream.get_final_completion()
#     print(final_response)

print("TEXT:", text2)
print(response)#.output_parsed)

try:
    # parsed_output = json.loads(response.output_text)
    # parsed_output = json.loads(response.to_json())["choices"][0]["message"]["content"]
    parsed_output = json.loads(response.to_json())["output"][1]["content"][0]["parsed"]["impact_cases"]
    # json.loads(response.to_json())["output"][1]["content"][0]["text"].split("ANSWER:")[1] 
    # print(parsed_output)    
    df_resp = pp.postprocess_response(parsed_output)

except Exception as e:
    print("Could not parse response as JSON:", e)
    pass
#     # parsed_output = None
#     for output in response.output:
#         if output.type != "message":
#             continue

#         for item in output.content:
#             if item.type == "refusal":
#                 # If the model refuses to respond, you will get a refusal message
#                 print(item.refusal)
#                 continue

#             if not item.parsed:
#                 raise Exception("Could not parse response")
            
#             print(item.parsed)

df_resp

In [ ]:
df_resp

#### memory and model shredding 

In [ ]:

# #
# import os

# # # settings for CUDA and PYTORCH
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # # settings for debugging CUDA errors (pinpoint exact line of error)
# os.environ["TORCH_USE_CUDA_DSA"] = "1"
# # os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# # activate global venv explicitly
# os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


# import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())  # should give 2
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])


## # settings for distributed computing
# NOTE find MASTER_ADDR and MASTER_PORT  (any unused port)
# hostname -I | awk '{print $1}'
# comm -23 <(seq 29500 30000 | sort) <(ss -Htan | awk '{print $4}' | cut -d: -f2 | sort -u) | head -n 1
os.environ["WORLD_SIZE"]="1"
os.environ["RANK"]="0"
os.environ["LOCAL_RANK"]="0"
os.environ["MASTER_ADDR"]="localhost" # 10.10.0.2
os.environ["MASTER_PORT"]="29500" # 12355


from torch import distributed as dist
# from transformers import AutoTokenizer, AutoModelForCausalLM

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
dist.init_process_group(backend="nccl")

env_dist = os.environ
print('env MASTER_ADDR={}'.format(env_dist['MASTER_ADDR']))
print('env MASTER_PORT={}'.format(env_dist['MASTER_PORT']))
print('env WORLD_SIZE={}'.format(env_dist['WORLD_SIZE']))
print('env RANK={}'.format(env_dist['RANK']))
print('env LOCAL_RANK={}'.format(env_dist['LOCAL_RANK']))



In [ ]:
model_name = "openai/gpt-oss-20b"  # bit bette performance than llama-3-70B but much smaller [43 GB VRAM]
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, its location type (e.g., region, county, city, river etc.), the type of damage and potentially quantitative information about the damage."
text = " In Berlin a storm caused major damages to the power grid and railway network"
# context = [{
#         "text": text,
#         "entity_recognition": None,}]


# ## init LLM extraction models
# decoder_model_1 = DecoderModelCaching(
#     model_name,
#     static_prompt=load_prompt_template(template_filename="short_static_llama3_NER.txt",)
# )


print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
print(os.environ["CUDA_VISIBLE_DEVICES"])

# load LLM_1 prompt template
static_prompt = load_prompt_template(template_filename="short_static_llama3_NER.txt")
dynamic_prompt = load_prompt_template(template_filename="short_dynamic_llama3_NER.txt")


################### init model + tokenizer

# try: 
#     login(token=os.getenv("HUGGINGFACE_TOKEN"))   # notebook_login
# except:
#     login(token=os.environ.get("HUGGINGFACE_TOKEN"))  # former HF_TOKEN


base_dir = s.HF_HOME_DIR   # use default dir in .cache/
model_dir = Path(base_dir)  # / f"models--{model_name.replace("/", "--")}"  # is already .._mirror/hub/
print(f"Model directory: {model_dir}")


# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

device_map = {
    # Enable Expert Parallelism
    "distributed_config": DistributedConfig(enable_expert_parallel=1),
    # Enable Tensor Parallelism
    "tp_plan": "auto",
} 

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=model_dir,
    local_files_only=True, 
    dtype="auto",  # torch.float32
    attn_implementation="kernels-community/vllm-flash-attn3",
    **device_map,
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    device_map="auto",
    cache_dir=model_dir,  # use fast Rust-based tokenizer, when possible
)
## reduce memory usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.use_checkpointing = True

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

################# # apply LLM 1 ###########################################
# response = decoder_model_1.generate_response(
#     question=question_1, context=context, # chunk_id=j
#     static_prompt = static_prompt,
#     dynamic_prompt=dynamic_prompt,
#     max_new_tokens = 2048
# )



In [ ]:
messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": f"""Try to answer following question `Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, the type of damage, and if given the quantitative information about the damage.` 
                                 based on the provided context: {user_content}"""},
    ]

#   static_prompt = static_prompt.render()
#     dynamic_prompt = dynamic_prompt.render(
#         context=context,  # includes also df_ci_geo info
#         question=question,
#     )

# clean up before applying CUDA
gc.collect()
# torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()


In [ ]:
messages = [
    {"role": "user", "content": "Explain what MXFP4 quantization is."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)#.type)

# # ORG class
# static_prompt = static_prompt.render()
# dynamic_prompt = dynamic_prompt.render(
#     context=context,  # includes also df_ci_geo info
#     question=question,
# )
# new_inputs = tokenizer(static_prompt + dynamic_prompt, return_tensors="pt").to(model.device.type)



# print("Using iterative caching of prompt key values to avoid recomputing entire prompt for each generation step.")
# # print("Using offloading currently") # to CPU for prompt cache to reduce GPU memory usage")
# prompt_cache =  DynamicCache(config=model.config) #, offloading=True) 
# static_prompt = static_prompt.render()
# inputs_initial_prompt = tokenizer(static_prompt, return_tensors="pt").to(model.device.type)
# # This is the common prompt cached, we need to run forward without grad to be able to copy
# with torch.no_grad():
#     prompt_cache = model(**inputs_initial_prompt, past_key_values=prompt_cache).past_key_values
# # print("--> Using currently offloading and only shallow copy of pk_values")
# # past_key_values = copy.copy(self.prompt_cache)  
# # print("--> Not using offloading, but deep copy of pk_values")
# past_key_values = copy.deepcopy(prompt_cache)  # Needed to copy past KV values
# # # FIXME - potential issues as pk_v is not copied as completely independent object (set offloading=True) 


# outputs = model.generate(**inputs, max_new_tokens=1024)
outputs = model.generate(
    **inputs, 
    # **new_inputs, 
    use_cache=True,  # MAYBE instead of past_key_values
    temperature=0.1,
    num_beams=1,  # test for better results, but slower generation (more exhaustive search for best output sequence)
    early_stopping=True, 
    # past_key_values=past_key_values, 
    do_sample=False,  # False, beam_nums=1  greedy search - default decoding strategy, good for short output and no creativity
    remove_invalid_values=True,
    max_new_tokens=2048,
    pad_token_id=tokenizer.eos_token_id
    )

# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1] :]))


In [ ]:
# print(tokenizer.batch_decode(outputs)[0])
# print(tokenizer.decode(outputs[0]))


In [ ]:
max_length = 1024
# input_str = "Provide a list of five words that are similar to the phrase: 'Knights who say Nie'"
text = " In Berlin a storm caused major damages to the power grid and railway network"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     cache_dir=model_dir,
#     local_files_only=True, 
#     dtype="auto",  # torch.float32
#     attn_implementation="kernels-community/vllm-flash-attn3",
#     **device_map,
# )
# tokenizer = AutoTokenizer.from_pretrained(
#     model_name,
#     use_fast=True,
#     device_map="auto",
#     cache_dir=model_dir,  # use fast Rust-based tokenizer, when possible
# )
# input_ids = tokenizer.apply_chat_template(
#             messages,
#             add_generation_prompt=True,
#             return_tensors="pt",
#             return_dict=True,
#         ).to(model.device),#.typ
# input_ids = tokenizer.encode(input_str, return_tensors='pt').to(model.device)
input_ids = tokenizer.apply_chat_template(
    messages,
    # add_generation_prompt=True,
    return_tensors="pt",
    # return_dict=True,
).to(model.device.type)

attention_mask = torch.ones(input_ids.shape, dtype=torch.long)
pad_token_id = tokenizer.eos_token_id


with torch.no_grad():
    outputs = model.generate(
        i, 
            use_cache=True,  # MAYBE instead of past_key_values
            temperature=0.1,
            # num_beams=1,  # test for better results, but slower generation (more exhaustive search for best output sequence)
            early_stopping=True, 
            # past_key_values=past_key_values, 
            # do_sample=False,  # False, beam_nums=1  greedy search - default decoding strategy, good for short output and no creativity
            remove_invalid_values=True,
        max_length=max_length, 
        # attention_mask=attention_mask, 
        pad_token_id=pad_token_id, do_sample=True)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto",
   # Flash Attention with Sinks
   attn_implementation="kernels-community/vllm-flash-attn3",
)

# messages = [
#     {"role": "user", "content": "How many rs are in the word 'strawberry'?"},
# ]
messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"Here is a text snippet from a document: {text2}. Answer follwing question: {question_1}"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)#.to(model.device.type)

generated = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))

## ERRORs  distb computing
# to(model.device.type)
# AcceleratorError: CUDA error: device-side assert triggered
# Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
## Also, issue of loading of files chekcpoiints takes a few secs

In [ ]:
# print(tokenizer.decode(outputs[0]))#[inputs["input_ids"].shape[-1] :]))
tokenizer.decode(outp[0]).message

# # inputs = tokenizer.apply_chat_template(
# #     messages,
# #     add_generation_prompt=True,
# #     return_tensors="pt",
# #     return_dict=True,
# # ).to(model.device)
# # # new_inputs = tokenizer(static_prompt + dynamic_prompt, return_tensors="pt").to(self.model.device.type)

# print(inputs)

# # outputs = model.generate(**inputs, max_new_tokens=1024)
# outputs = model.generate(
#     **inputs, 
#     # **new_inputs, 
#     use_cache=True,  # MAYBE instead of past_key_values
#     temperature=0.1,
#     num_beams=1,  # test for better results, but slower generation (more exhaustive search for best output sequence)
#     early_stopping=True, 
#     # past_key_values=past_key_values, 
#     do_sample=False,  # False, beam_nums=1  greedy search - default decoding strategy, good for short output and no creativity
#     remove_invalid_values=True,
#     max_new_tokens=1024,
#     pad_token_id=tokenizer.eos_token_id
#     )


In [ ]:
env_dist = os.environ
# print('env MASTER_ADDR={}'.format(env_dist['MASTER_ADDR']))
# print('env MASTER_PORT={}'.format(env_dist['MASTER_PORT']))
print('env WORLD_SIZE={}'.format(env_dist['WORLD_SIZE']))
print('env RANK={}'.format(env_dist['RANK']))
print('env LOCAL_RANK={}'.format(env_dist['LOCAL_RANK']))


In [ ]:
# # settings for distributed computing
# os.environ["WORLD_SIZE"]="1"
# os.environ["RANK"]="0"
# os.environ["LOCAL_RANK"]="0"
# os.environ["MASTER_ADDR"]="localhost" # 10.10.0.2
# os.environ["MASTER_PORT"]="12355" # 29500

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# from torch import distributed as dist
# from transformers import AutoTokenizer, AutoModelForCausalLM

# local_rank = int(os.environ["LOCAL_RANK"])
# torch.cuda.set_device(local_rank)
# dist.init_process_group(backend="nccl")

# # NOTE find MASTER_ADDR and MASTER_PORT  (any unused port)
# # hostname -I | awk '{print $1}'
# # comm -23 <(seq 29500 30000 | sort) <(ss -Htan | awk '{print $4}' | cut -d: -f2 | sort -u) | head -n 1


## # settings for distributed computing
# NOTE find MASTER_ADDR and MASTER_PORT  (any unused port)
# hostname -I | awk '{print $1}'
# comm -23 <(seq 29500 30000 | sort) <(ss -Htan | awk '{print $4}' | cut -d: -f2 | sort -u) | head -n 1
os.environ["WORLD_SIZE"]="1"
os.environ["RANK"]="0"
os.environ["LOCAL_RANK"]="0"
os.environ["MASTER_ADDR"]="localhost" # 10.10.0.2
os.environ["MASTER_PORT"]="12355" # 29500


from torch import distributed as dist
# from transformers import AutoTokenizer, AutoModelForCausalLM

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
dist.init_process_group(backend="nccl")



In [ ]:
# !hf cache scan

## Run

In [ ]:
# from scalene import scalene_profiler
# scalene_profile_2 = scalene_profiler


In [ ]:
# %%scalene --reduced-profile  
# # # Turn profiling on
# scalene_profiler.start()



# Settings
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "meta-llama/Meta-Llama-3-70B-Instruct"  # [144 GB VRAM]
model_name = "openai/gpt-oss-20b"  # bit bette performance than llama-3-70B but much smaller [43 GB VRAM]


# #gguf_filename = "models--unsloth--Llama-4-Maverick-17B-128E-Instruct-GGUF"#"Llama-4-Maverick-17B-128E-Instruct-UD-Q4_K_XL.gguf"
# model_name = "unsloth/Llama-4-Maverick-17B-128E-Instruct-GGUF"
# gguf_filename = "Llama-4-Maverick-17B-128E-Instruct-UD-Q4_K_XL.gguf"


time0 = time.time()

# clean up before applying CUDA
gc.collect()
torch.cuda.empty_cache() 
print(torch.cuda.memory_reserved() / 1e9)
torch.no_grad()

print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
print(os.environ["CUDA_VISIBLE_DEVICES"])


# Questions
question_1 = "Which infrastructure failures are mentioned in the text? Categorize the output by the type of infrastructure, the location, its location type (e.g., region, county, city, river etc.), the type of damage and potentially quantitative information about the damage."
question_2 = "Is the location of each affected or damaged critical infrastructure correctly identified?"


## init LLM extraction models
decoder_model_1 = OpenAIModel(
    model_name,
    system_prompt=em.load_prompt_template(template_filename="gpt_system_prompt.txt")
)
decoder_model_2 = OpenAIModel(
    model_name,
    system_prompt=em.load_prompt_template(template_filename="gpt_system_prompt_step2.txt")
)

# decoder_model_1 = em.DecoderModelCaching(
#     model_name,
#     static_prompt=em.load_prompt_template(template_filename="short_static_llama3_NER.txt",)
# )

# decoder_model_2 = em.DecoderModelCaching(
#     model_name,
#     static_prompt=em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt",)
# )


# load tokenizer
embed_model =  "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(embed_model),
    max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
    # standardize input sizes of chunks for Llama models
    padding=True, # add zero as extra tokens to too short sequences so that they have the same length as other chunks
    truncation=True, # truncates too long sequences (> max_tokens). If False, they will be split into multiple chunks
)

## init chunker - based on hierachical chunker but also considers max token leng, merge smaller chunks, except when at end of paragraph (merge_peers=True)
chunker = HybridChunker(
    tokenizer=tokenizer,
        # max_tokens=256, # max tokens for MiniLM-l6-v2, set here explicitly
        # chunk_overlap=0, # no overlap between chunks, as we use merge_peers to merge smaller chunks and avoid splits in sentences
    split_by_sentence=True, # split by sentence first before merging smaller chunks, to avoid splits in sentence middle
    merge_peers=True,  # optional, defaults to True
)

## init NER model pipe
nlp = NERModel().nlp_model



gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)

# print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# print(os.environ["CUDA_VISIBLE_DEVICES"])

# for Doc parsing and cleaning
md_converter = DocumentConverter(allowed_formats=[InputFormat.MD])


# CI-GEO pairs
geolocs_cache = geonamescache.GeonamesCache()
countries = geolocs_cache.get_countries()
ci_geo_countries = [*u.gen_dict_extract(countries, 'name')]



## init outputs
df_resp_step1_all = pd.DataFrame()
df_resp_step2_all = pd.DataFrame()
responses_error_list = [] 
df_ci_cases_not_grouped = pd.DataFrame()
df_geollama_response = pd.DataFrame()


if test_mode:
    search_path = docs_list_sample
    print("Test mode is ON. Using only a small sample of documents for testing.")
else:
    search_path = glob(str(Path(PARSED_TEXT_DIR, "*cleaned.jsonl")))




## Start CI impact extraction
for file_no, filename in enumerate(search_path):

    EXPORT_TYPE = ExportType.DOC_CHUNKS
    mislabeled: List[DocItem] = []
    min_paragraph_size = 50  
    temp_docs: List[ByteStream] = []
    temp_meta: List[Dict[str, str]] = []
    i: int
    combined_paragraph: str = ""
    combined_chars: int = 0
    para_num: int = 0
    section_name: str = ""
    page_no: Optional[int] = None
    first_note: bool = False


    time1 = time.time()

    src_language_nonengl = None 

    no_documents = len(search_path)
    filepath = Path(filename)
    filename_stem = filepath.stem


    print(f"\n\n ######## -------- Processing document [{file_no+1}/{no_documents}]: {filepath.name} -------- ######## \n")

    ## extract authors, publication year and title 
    author, year, title = dc.extract_citation_info(filename_stem)
    citation = f"{author} {year}".replace("  ", " ").strip()
    title = title.replace(" - ", "").replace("_cleaned", "").strip()



    # init dfs to store interim results for each doc
    df_resp_step1 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "damage",
            "damage_value",
            "location",
            "location_type",
            "ci_entity",
            "geo_entity",
            "chunk_text"
        ]
    )
    df_resp_step2 = pd.DataFrame(
        columns=[
            "citation_id",
            "chunk_id",
            "infrastructure_type",
            "infrastructure_group",
            "damage",
            "damage_value",
            "location",
            "location_type",
            "ci_entity",
            "geo_entity",
            "coord_potential_locations",
            "chunk_text"
        ]
    )
    
    print(f"\n ##### ------- Cleaning document -----------########")

    pdf_filepath = os.path.join(DOCS_DIR, Path(filename_stem + ".pdf"))
    md_filename = filename_stem + ".md"
    md_filepath = os.path.join(PARSED_TEXT_DIR, Path(md_filename))
    cleaned_md_filepath = md_filepath.replace(".md", "_cleaned.md")
    # cleaned_jsonl_filepath = md_filepath.replace(".md", "_cleaned.jsonl")

    time_cleaning = time.time()

    if os.path.exists(cleaned_md_filepath):
        print(f"Cleaned markdown file already exists, loading file and proceeding with Ci impact extraction")
        # Load the existing cleaned markdown file
        doclingdoc = md_converter.convert(cleaned_md_filepath).document      

    elif os.path.exists(pdf_filepath):
        print(f"Cleaned markdown file does not exist yet. Cleaning document by using original PDF: '{filename}'")

        # get language of document
        src_language_doc = langdetect.detect(filename_stem.lower())  # lower case improves language detection


        ## Document converter with OCR
        print("Using OCR for text extraction as it identifies section titles, footers/headers and pagenumbers as such, but reads in also figure text sometimes") 
        # NOTE all other standard doclingConverter retunr section/headers etc as BODY not FURNITURE
        # NOTE: partly reads in figure text and table text 
        pdf_doc_org = dc.DocumentParser().ocr_converter.convert(source=pdf_filepath).document

        ##  get only list of Doc.items
        texts = dc.get_processed_texts(pdf_doc_org) 

        texts_clean = []
        section_names = []

        # text cleaning, annotating section names, remove header/footers, and translation
        for i, text in enumerate(texts):
            

    ### TODO make as Parser Doc class
            # get next text only when it is not page header/footer
            next_text = dc.get_next_text(texts, i)
            # page_no = get_current_page(text, combined_paragraph, page_no)


            # Update section header if the element is a section header
            # TODO: Need a stronger check on section headers that takes top of page into account, etc
            if dc.is_section_header(text) and text not in mislabeled:
                print("!!  Section header found:", text.text)
                section_name = text.text
                continue

            if dc.is_reference_section(section_name):
                print("Reference section found. Stopping further processing of document.")
                break  

            if dc.should_skip_element(text):
                continue
            
            # clean from double whitespace, newlines, etc.
            p_str = dc.clean_text(text.text)

            # clean from potential figure references
            p_str = dc.remove_figure_references(p_str)

            ## replace e.g. and i.e. --> eg and ie to avoid sentence splits
            p_str = re.sub(r"e\.g\.\s+", "eg ", p_str)
            p_str = re.sub(r"i\.e\.\s+", "ie ", p_str)

            # Removing URLs 
            # LangExtract tries to open these URLs when they occur in the document text
            # p_str= re.sub(r"http\S+", "", p_str) 

            p_str_chars = len(p_str)

            # If the paragraph does not end with final punctuation, accumulate it
            if not dc.is_sentence_end(p_str):
                combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_chars += p_str_chars
                continue

            # p_str ends with a sentence end; decide whether to process or accumulate it
            total_chars = combined_chars + p_str_chars
            if dc.is_section_header(next_text):
                # Immediately process if the next text is a section header
                p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_paragraph, combined_chars = "", 0
            elif total_chars < min_paragraph_size:
                # Not enough characters accumulated yet; decide based on next_text
                if next_text is None or (not dc.is_page_text(next_text) and dc.is_sentence_end(p_str)):
                    # End of document or next text item is not a text item and current paragraph ends with punctuation
                    # Process the paragraph and reset the accumulator even though this is a short paragraph
                    p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                    combined_paragraph, combined_chars = "", 0
                else:
                    # Combine with next paragraph
                    combined_paragraph = dc.combine_paragraphs(combined_paragraph, p_str)
                    combined_chars = total_chars
                    continue
            else:
                # Sufficient characters: process the paragraph and reset the accumulator
                p_str = dc.combine_paragraphs(combined_paragraph, p_str)
                combined_paragraph, combined_chars = "", 0

            p_str = dc.combine_hyphenated_words(p_str)
            if p_str:  # Only add non-empty content
                para_num += 1
                dc.add_paragraph(
                    p_str, 
                    #para_num, section_name, page_no, 
                    temp_docs, 
                    # temp_meta
                )
                page_no = None
            
            print("\n-__ Paragraph #", para_num, "; Section:", section_name)

            # print(p_str)
            # print(temp_docs, temp_meta)

### TODO translator class
            ## Translation
            if src_language_doc != "en":
                try:
                    supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
                except Exception as e: 
                    print("! Exception: translaton: cant load for itc language ")
                    supported_languages = ["fr", "de", "es", "it", "nl"]
                # supported_languages = ["en", "fr", "de", "es", "pt", "it", "pl", "cs", "nl", "da", "sv", "no", "hr", "ro", "bg", "sl", "sk", "lt", "et" ],
                if src_language_doc not in supported_languages:
                    print(f"Unsupported source language: {src_language_doc}. Continue with extraction on original text")
                    continue 

                print(f"Translating {src_language_doc} --> en")
                try:
                    p_str = tm.translate_2_english(src_language_doc, p_str)     
                except Exception as e:
                    print(f"! Cannot translate text, going to next chunk: {p_str}")
                    continue

            # collect cleaned text + meta data per doc            
            texts_clean.append(p_str)
            section_names.append(section_name)


        print("TEST Saving cleaned text as DoclingDocument to store text (p_str)+ meta (section title, page_no): ")
        # Initialize new DoclingDoc
        doclingdoc = DoclingDocument(schema_name="DoclingDocument",  version="1.0.0",name="My Custom Document")
        
        # write text and structural elements (titles, sections, and paragraphs) to doc     
        title_node = doclingdoc.add_title(text=filename_stem)
        for text, section_name in zip(texts_clean, section_names):
            print(section_name, ":", text)
            section_node = doclingdoc.add_heading(text=section_name, level=1, parent=title_node)
            doclingdoc.add_text(label=DocItemLabel.TEXT, text=text, parent=section_node)

        # Save doclingDocument as MD
        doclingdoc.save_as_markdown(cleaned_md_filepath) 

    # no pdf or md exists, skipping document
    else:    
        print(f"Neither cleaned markdown file nor original pdf file exists for document: '{filename}'. Skipping document.")
        continue

    print(f"Document cleaning took, {np.round((time.time() - time_cleaning) / 60, 1)} minutes")


    print("Chunking document...")
    chunk_iter = chunker.chunk(dl_doc=doclingdoc)  # NOTE cannot use chunker when re-created Docl.Document with cleaned text and old DoclingObject (from converter)
    doc = list(chunk_iter)
    # print(len(chunks), "chunks created with chunker.chunk(dl_doc=doclingdoc)")
    # print(len(texts_clean), "cleaned text items in doclingdoc.texts")
    
    # TODO 
    # test if contextualization improves model performance 

    # # apply contextualization (add section_name etc to chunk textfor better LLM unterstanding)
    # for i in range(len(chunks)):
    #     print(f"\nChunk {i} content before contextualization:\n{chunks[i].text}")
    
    #     ser_text = chunker.contextualize(chunk=chunks[i])
    #     ser_tokens = tokenizer.count_tokens(ser_text)
    #     print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_text!r}")
    #     doc = ser_text
    #     print()
    

    
    for chunk_no, chunk in enumerate(doc):

        # scalene_profile_2.start()


          

        # init dfs for interim results for each chunk
        ## TODO make as pydantic class with fixed attributes
        # df_ci_geo_chunk = pd.DataFrame(
        #     columns=[
        #         "citation_id",
        #         "chunk_text",
        #         "ci_entity",
        #         "ci_entity_label",
        #         "geo_entity",
        #         "geo_entity_label",
        #         "token_distance",
        #     ]
        # )    
    
        df_geollama_response = pd.DataFrame()

    
        print(f"\nProcessing chunk no. {chunk_no+1} / {len(doc)} of document: {filepath.name}")
        print("Chunk text: ", chunk.text)


        ## add punctuations back for e.g. and i.e. (for better reading in chunks by LLm-WF)
        chunk.text = re.sub(r"eg\s+", "e.g. ", chunk.text)
        chunk.text = re.sub(r"ie\s+", "i.e. ", chunk.text)

        ## FIXME TODO fix wrongly set hyphens
        chunk.text = re.sub("SachsenAnhalt", "Sachsen-Anhalt", chunk.text)
        

        

        # print("Getting geolocations of CI assets ")       
        # ## get most likely geolocation for each CI entity based on distance between tokens
        # nlp_chunk = nlp(chunk.text)
        # all_ents = [ent for ent in nlp_chunk.ents]
        # ci_type_ents = [ent for ent in nlp_chunk.ents if ent.label_ in ["CI_TYPE"]]#, "FAC"]]


        # # check if chunk contains CI_TYPE entities
        # if len(ci_type_ents) > 0:

        #     # iterate over all entities within chunk
        #     for ent_idx in range(len(all_ents)):
        #         # when entity is CI_TYPE or FAC (i.e. buidling, airports, highways) do following ...
        #         if all_ents[ent_idx].label_ in ["CI_TYPE"]: #, "FAC"]:
        #             ci_idx = ent_idx

        #             ## .. calculate distances between CI_TYPE entity and  all GEO entities in chunk based on index position
        #             distance_list = []
        #             idx_in_chunk = []
        #             try:
        #                 for ent_idx in range(len(all_ents)):

        #                     # TODO calc distances between CI_TYPE ~ GEO entities based on word numbers and not entities (ie tokens)
        #                     if all_ents[ent_idx].label_ in ["GPE", "LOC"]:

        #                         ## check that GPE,LOC are not countries (too coarse info CI-GEO pair)
        #                         # of GPE/LOC is country -> proceed with next GPE/LOC 
        #                         # if all_ents[ent_idx].text in ci_geo_countries:
        #                         #     continue

        #                         geo_idx = ent_idx
        #                         dist_ent_pair = np.abs(ci_idx - geo_idx)
        #                         distance_list.append(dist_ent_pair)
        #                         idx_in_chunk.append((ent_idx))
        #                         closest_pair_idx = np.argmin(distance_list)  # idx of closest GEO entity
        #                         distance_closest_pair = distance_list[closest_pair_idx]

        #                 threshold = 10  # max token distance between CI_TYPE and GEO entity
        #                 if distance_closest_pair > threshold:
        #                     # print(
        #                     #     f""" Token distance is too large between CI_TYPE/FAR and next GEO entity which is of {distance_closest_pair} [token distance] > {threshold} [max. token distance] """
        #                     # )
        #                     continue
        #                 else:
        #                     pass

        #                 ## write as dict entry incl chunk_id, ci_entity, geo_entity, distance
        #                 result_dict = {
        #                     "citation_id": citation,
        #                     "chunk_text": chunk.text,
        #                     "ci_entity": all_ents[ci_idx].text,
        #                     "ci_entity_label": all_ents[ci_idx].label_,
        #                     "geo_entity": all_ents[idx_in_chunk[closest_pair_idx]].text,
        #                     "geo_entity_label": all_ents[idx_in_chunk[closest_pair_idx]].label_,
        #                     "token_distance": distance_closest_pair,
        #                 }
        #                 df_ci_geo_chunk = pd.concat(
        #                     [  df_ci_geo_chunk, pd.DataFrame([result_dict])], ignore_index=True
        #                 )

        #             except (IndexError, NameError) as e:
        #                 continue
        # else:
        #     print("No CI_TYPE entities found in this chunk. Going to next chunk")
        #     continue

        
        # ## post-process of DF CI-GEO pairs for each chunk
        # unique_ci_geo_pairs = df_ci_geo_chunk.drop_duplicates(
        #     subset=["citation_id", "ci_entity", "geo_entity","chunk_text"])
        # print("number of duplicates to remove:", len(df_ci_geo_chunk) - len(unique_ci_geo_pairs))

        # df_ci_geo_chunk = df_ci_geo_chunk.drop_duplicates(
        #     subset=["citation_id", "ci_entity", "geo_entity", "chunk_text"]
        #     )# .reset_index(drop=True, inplace=True)


        print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        print(os.environ["CUDA_VISIBLE_DEVICES"])

        print("\nSTEP 1")

        ## apply decoder on each chunk in document
        ## TODO replace iteration by loading entire document and use recursive chunking from langchain
        time2 = time.time()


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        torch.no_grad()
        # print(torch.cuda.memory_reserved() / 1e9)

        print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        print(os.environ["CUDA_VISIBLE_DEVICES"])

        # load LLM_1 prompt template
        # user_static_prompt = load_prompt_template(template_filename="short_static_llama3_NER.txt")
        # dynamic_prompt = load_prompt_template(template_filename="short_dynamic_llama3_NER.txt")
        user_prompt = load_prompt_template(template_filename="gpt_user_prompt.txt")

        # pass dynamic content from chunk
        context = [
            {
                "text": chunk.text,
                "citation": citation,
                "title": filename_stem, 
            },
        ]
        user_prompt = user_prompt.render(
                context=context,
                question=question_1,
        )
        user_prompt =  f"""You will be given an unstructured text from newspaper articles and research papers called CONTEXT.
            For each case of damage or impact on critical infrastructure you should convert it into the given structure.
            Try to answer following question {question_1} by following the step-by-step instructions in the PROMPT.
            PROMPT and CONTEXT: {user_prompt};
            """

        # apply LLM 1
        time_llm1 = time.time()
        try:
            response = decoder_model_1.generate_response_with_tool(
                max_new_tokens = 2048,
                user_prompt = user_prompt,
                chunk_text = context[0]["text"],
                # user_dynamic_prompt: load_prompt_template,     
            )            
        except Exception  as e: 
            print(f"! ERROR {e}. Sleeping for three minute and then try again")
            time.sleep(120)
            try: 
                response = decoder_model_1.generate_response_with_tool(
                    max_new_tokens = 2048,
                    user_prompt = user_prompt,
                    chunk_text = context[0]["text"],
                    # user_dynamic_prompt: load_prompt_template,     
                )
            except Exception as e:
                print(f"! Could not call model going to next chunk: {e}")
                continue
        print(f"calling LLM 1 took {(time.time() - time_llm1) / 60} min" )
        

        ## postprocess response
        try:
            # GPT-OSS + tool NER
            if isinstance(response[0], dict):
                df_resp = pd.DataFrame(response)
            #if isinstance(json.loads(response.to_json()), dict):
            #    df_resp = pd.DataFrame(response)
            # models without tools
            # Llama models
            if isinstance(response, dict, ):
                df_resp = pp.postprocess_response(response)
            # GPT-OSS
            if isinstance(response[0], (str, TypeVar)):
                df_resp = pp.postprocess_response(response[0])
           
            # check if df is empty
            if "infrastructure_type" not in df_resp.columns:
                continue
            

            # enrich LLM response (per chunk)
            df_resp["citation_id"] = citation 
            df_resp["chunk_id"] = chunk_no  # add chunk id as identifier
            # df_resp["ci_entity"] = df_ci_geo_chunk["ci_entity"] 
            # df_resp["geo_entity"] = df_ci_geo_chunk["geo_entity"]
            # df_resp["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 
            df_resp["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response

            if not len(df_resp):
                print("LLM response is empty. Continue with next chunk.\n Response was:", response)
                continue

            # # remove infrastructure_types which not really occur in chunk
            ## NOTE: would also remove "metro" whe ntext is about certain subway stations
            # for case_no in df_resp.index:
            #     smlrty = fuzz.partial_ratio(df_resp["infrastructure_type"][case_no], df_resp["chunk_text"][case_no])

            #     # remove CI case when it is not in chunk
            #     if smlrty < 70:   #  eg. 73: railway infrastructure <-> This is some text about railways that are infrastructure with damages
            #         print(f"! CI type {df_resp["infrastructure_type"]} (partial ratio) does not occur in chunk text. Removing case")
            #         df_resp = df_resp[df_resp["infrastructure_type"] != df_resp["infrastructure_type"][case_no]]
            #         continue

            # store response for chunk to interim df
            df_resp_step1 = pd.concat([df_resp_step1, df_resp], ignore_index=True)
    

        except (IndexError, ValueError, KeyError) as e:
            print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
            # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
            responses_error_list.append({
                "citation_id": citation,
                "chunk_id": chunk_no,
                "response": response,
                "error": str(e)
            })
            print("Continue with next chunk\n")
            continue



        ## group Ci types into subgroups, 
        if df_resp.empty == True:
            print("No infrastructure types extracted for this chunk, skip grouping into subgroups. Go to next chunk")
            continue

        df_resp["infrastructure_type"] = df_resp["infrastructure_type"].replace(r"[\(\),]", "", regex=True)   # clean up CI from brackets and commas
        
        if "infrastructure_group" in df_resp.columns:
            df_resp = pp.group_ci_types(df_resp, "infrastructure_type", "infrastructure_group", ci_patterns)
        else:
            df_resp["infrastructure_group"] = None            
            df_resp = pp.group_ci_types(df_resp, "infrastructure_type", "infrastructure_group", ci_patterns)
        
        ## store cases which could not be grouped
        df_ci_cases_not_grouped = pd.concat([df_ci_cases_not_grouped, df_resp[df_resp['infrastructure_group'].isna()]], ignore_index=True)
        ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
        df_resp.dropna(subset=["infrastructure_group"], inplace=True)
    

        # clean up after each chunk
        gc.collect()
        torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
        torch.no_grad()

        print(f"### ---- Processing time for chunk {chunk_no+1} STEP 1: {np.round((time.time() - time2) / 60, 1)} minutes ---- ###")


        print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        print(os.environ["CUDA_VISIBLE_DEVICES"])

        
        print("\nGEOLLAMA")

        print(f"Starting geoparsing")

        time_geoparse = time.time()

        ## Start geoparsing with geollama to verify Location        
        
        # extract locations
        # FIXME             
        ## replace e.g. and i.e. --> eg and ie to avoid sentence splits
        chunk.text = re.sub(r"e\.g\.\s+", "eg ", chunk.text)
        chunk.text = re.sub(r"i\.e\.\s+", "ie ", chunk.text)


        ## TEST if geollama finds more LOCs (eg. rivers) when applied on sentence level
        sentences = []
        df_geollama_resp = pd.DataFrame()
        for s in chunk.text.split(". "):
            ## aaply geollama on sentences and append response to DF
            resp = geo_llama.geoparse(s)
            df_geollama_resp = pd.concat([df_geollama_resp, pd.DataFrame(resp[0:])], ignore_index=True)
            sentences.append(s)
        print(f"Time for geoparsing took {np.round((time.time() - time_geoparse) / 60, 1)}" )

        ## merge single sentences back and add punctuations back for e.g. and i.e.
        chunk.text = ". ".join(sentences)
        chunk.text = re.sub(r"eg\s+", "e.g. ", chunk.text)
        chunk.text = re.sub(r"ie\s+", "i.e. ", chunk.text)

        # add metadata to geollama response per chunk
        df_geollama_resp["citation_id"] = citation
        df_geollama_resp["chunk_id"] = chunk_no
        df_geollama_resp["chunk_text"] = chunk.text

        # maybe empty response
        try:
            # print("Removing duplicated locations and countries from geollama response to keep only more specific location info")
            print("Removing duplicates, and keep countries in the geollama response")

            df_geollama_resp["name"] = list(set(df_geollama_resp["name"]))  # rm dublicates
            # df_geollama_resp = df_geollama_resp[~df_geollama_resp["name"].isin(ci_geo_countries + ["days", "months", "years", "Netherlands"])]  # rm countries
            print("Toponyms from geollama\n", df_geollama_resp["name"])
        except:
            pass

        # saving all geollama repsonses , even when they cannot processed furthernfor later analysis
        df_geollama_response = pd.concat([df_geollama_response, df_geollama_resp], ignore_index=True)
        

        try:
            locations_per_chunk = df_geollama_resp["name"].to_list()
            lats_per_chunk = df_geollama_resp["latitude"].to_list()
            lons_per_chunk = df_geollama_resp["longitude"].to_list()
            RAGestimated_per_chunk = df_geollama_resp["RAG_estimated"].to_list()
        except:
            print("No locations extracted by geollama for this chunk. Going to next chunk\n", resp[0:])
            continue
        
        # sanity check that only cases with location infos are considered for further processing      
        if len(locations_per_chunk) == 0:
            print("geollama did not find any potential locations found for this chunk. Continue with next chunk")
            continue

        
        print("Enriching LLm 1 response with GeoLlama locations")
        df_resp["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 



        print("\nSTEP 2")

        print("KEEPING location info from step 1 for further improvement")
        df_locs_org = df_resp.copy() 


        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        print(torch.cuda.memory_reserved() / 1e9)
        torch.no_grad()

        # print(os.environ["CUDA_VISIBLE_DEVICES"])
        # os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
        # print(os.environ["CUDA_VISIBLE_DEVICES"])


        ## Input for LLM - 2nd round (geollama)
    
        # print(f"\nCHECKING input.text, df_resp, geollama resp \n{chunk.text}\n{df_resp},\n{locations_per_chunk}")
        # static_prompt_geollama = em.load_prompt_template(template_filename="short_static_llama3_NER_geollm_step2.txt")
        user_prompt_geollama = em.load_prompt_template(template_filename="gpt_user_prompt_step2.txt")
        # dynamic_prompt_geollama = em.load_prompt_template(template_filename="short_dynamic_llama3_NER_geollm_step2.txt")
            
        context = [
            {
                "text": chunk.text,
                "citation": citation,
                "title": filename_stem, 
                "previous_response": df_resp,
                "potential_locations": locations_per_chunk,  # list of 1 or multiple location strings (no countries)      
                # "coordinates_potential_locations": list(zip(locations_per_chunk, lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)),
            },
        ]
        
        # pass dynamic content from chunk and previous LLM response
        user_prompt_geollama = user_prompt_geollama.render(
                context=context,
                question=question_2,
        )
                
        # LLM 1 - step2
        time_llm2 = time.time()
        try:
            response = decoder_model_2.generate_response(
                # question=question_2, # context=context, # chunk_id=j
                user_prompt = user_prompt_geollama,
                max_new_tokens = 2048,
            )
        except Exception as e: 
            print(f"! ERROR {e}. Sleeping for three minute and then try again")
            time.sleep(120)
            try: 
                response = decoder_model_2.generate_response(
                    max_new_tokens = 2048,
                    user_prompt = user_prompt,
                    # user_dynamic_prompt: load_prompt_template,     
                )
            except Exception as e:
                print(f"! Could not call model going to next chunk, {e}")
                continue
        
        # response = decoder_model_2.generate_response(
        #     question=question_2, context=context, # chunk_id=j
        #     static_prompt = static_prompt_geollama,
        #     dynamic_prompt = dynamic_prompt_geollama,
        #     max_new_tokens = 2048
        # )
        print(f"calling LLM 2 took {(time.time() - time_llm2) / 60} min" )
        

        ## postprocess response
        try:
            # models without tools
            # GPT-OSS
            if isinstance(response, str):
                df_resp_2 = pp.postprocess_response(response)
            #if isinstance(response[0], (str, list, TypeVar)):
            #    df_resp_2 = pp.postprocess_response(response[0])
            # Llama models
          

            # enrich LLM response for each chunk
            df_resp_2["citation_id"] = citation
            df_resp_2["chunk_id"] = chunk_no  # add chunk id as identifier
            # df_resp_2["ci_entity"] = df_ci_geo_chunk["ci_entity"] 
            # df_resp_2["geo_entity"] = df_ci_geo_chunk["geo_entity"]
            df_resp_2["coord_potential_locations"] = str(dict(zip(locations_per_chunk, zip(lats_per_chunk, lons_per_chunk, RAGestimated_per_chunk)))) # INTERIM for verification of lat, lon 
            df_resp_2["infrastructure_type_org"] = df_locs_org["infrastructure_type"] 
            df_resp_2["damage_org"] = df_locs_org["damage"] 
            df_resp_2["damage_value_org"] = df_locs_org["damage_value"] 
            df_resp_2["location_org"] = df_locs_org["location"] 
            df_resp_2["location_type_org"] = df_locs_org["location_type"] 
            df_resp_2["chunk_text"] =  context[0]["text"]  # add (translated) chunk text for tracing back LLM response

            # collect resps for each doc
            print("CREATED final LLM response (STEP 1 & 2) successfully")
            df_resp_step2 = pd.concat([df_resp_step2, df_resp_2], ignore_index=True)
            # print(df_resp_2)

        except (IndexError, ValueError, KeyError) as e:
            print(f"Cannot add response: {e},\nFaulty response (before postprocessing):", response)
            # faulty response: e.g.  .., "location": "V" on satellite and online on radar"}, { ...}, {}
            responses_error_list.append({
                "citation_id": citation,
                "chunk_id": chunk_no,
                "response": response, #.replace('\n', ''),
                "error": str(e)
            })

        print(f"\n   Processing time for chunk {chunk_no+1} STEPs 1 & 2: {np.round((time.time() - time2) / 60, 1)} minutes\n")

        # clean up before applying CUDA
        gc.collect()
        torch.cuda.empty_cache() 
        print(torch.cuda.memory_reserved() / 1e9)
        torch.no_grad()

        print(os.environ["CUDA_VISIBLE_DEVICES"])



    print(f"\n   Processing time for document {citation} STEP 1&2: {np.round((time.time() - time1) / 60, 1)} minutes\n")

    print(f"Safety: saving responses (Step 1 + 2) for doc: {citation} ")
    df_resp_step1.to_csv(f"./interim_results/llm_geollm_step1_{citation}.csv", encoding='utf-8', index=False)
    df_resp_step2.to_csv(f"./interim_results/llm_geollm_step2_{citation}.csv", encoding='utf-8', index=False)

    # clean up after each document
    gc.collect()
    torch.cuda.empty_cache()  # mainly needed after training, small effect when LLM applied only for inference
    torch.no_grad()

    # collecting all docs
    df_resp_step1_all = pd.concat([df_resp_step1_all, df_resp_step1], ignore_index=True)
    df_resp_step2_all = pd.concat([df_resp_step2_all, df_resp_step2], ignore_index=True)


    # # # Turn profiling off
    # scalene_profiler.stop()
    # scalene_profile_2.stop()
    

print(f"\n\n ######## -------- CI impact extraction took {(time.time() - time0) / 60} minutes -------- ######## \n\n")


# %%


# %% [markdown]
# ### Finish run

# %%
print("Where CI types could not be grouped:\n", df_ci_cases_not_grouped)


# %%

df_resp_step2.info()

# %%
print(len(df_resp_step1))
unique_ci_geo_pairs = df_resp_step1.drop_duplicates()
print("number of duplicates to remove:", len(df_resp_step1) - len(unique_ci_geo_pairs))

df_resp_step1_nodupl = df_resp_step1.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_resp_step1_nodupl))
df_resp_step1_nodupl

# %%
print(len(df_resp_step2))
unique_ci_geo_pairs = df_resp_step2.drop_duplicates()
print("number of duplicates to remove:", len(df_resp_step2) - len(unique_ci_geo_pairs))

df_resp_step2_nodupl = df_resp_step2.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_resp_step2_nodupl))
df_resp_step2_nodupl


print(df_resp_step2.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_resp_step2.infrastructure_group.value_counts()) # four most common subgroups seems to be correct
# df_pred.infrastructure_group.unique()



gc.collect()
torch.cuda.empty_cache() 
torch.no_grad()
print(torch.cuda.memory_reserved() / 1e9)
# %%
print(torch.cuda.memory_reserved() / 1e9)


# %%
print("Chunk with erroneous responses:", responses_error_list.__len__())
df_responses_error = pd.DataFrame(responses_error_list)
# df_responses_error#.tail(3)


# %%
# df_responses_all_step2#.tail(3)

# %% [markdown]
# ### Saving

# %%
# PATH_LLM_DATA: Path = Path(s.PATH_DATA /"llm_outputs/")
# LLM_DATA_FILENAME: str = "llm_1_updprompt_distanceNER.csv"

# OUTPUT_LLM1_FILEPATH =  Path(PATH_LLM_DATA / LLM_DATA_FILENAME)#.replace(".csv", "_v2.csv"))
# OUTPUT_LLM1_FILEPATH 

# %%
safety_df = df_resp_step2_all.copy()

# save LLM 1 output to disk along with prompt text
if not os.path.isfile(OUTPUT_LLM1_FILEPATH):

    print(f"Saving prompt, LLM response and erroneous responses [.txt, .csv] to {OUTPUT_LLM1_FILEPATH} ...")

    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(static_prompt.render(context=context, question=question_1))
    try:
        with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
            f.write(dynamic_prompt.render(context=context, question=question_1))
    except Exception as e:
        print("UndefinedError: dynamic_prompt has probably no df_ci_geo (it is empty)")   
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(static_prompt_geollama.render(context=context, question=question_2))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}.txt", "w") as f:
        f.write(dynamic_prompt_geollama.render(context=context, question=question_2))

    df_resp_step1_all.to_csv(f"{OUTPUT_LLM1_FILEPATH.stem}_step1.csv", encoding='utf-8', index=False)
    df_resp_step2_all.to_csv(f"{OUTPUT_LLM1_FILEPATH.stem}_step2.csv", encoding='utf-8', index=False)
    df_responses_error.to_csv(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}.csv", encoding='utf-8', index=False)

elif os.path.isfile(OUTPUT_LLM1_FILEPATH) and not os.path.isfile(OUTPUT_LLM1_FILEPATH.parent / f"{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"):

    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Saving as {OUTPUT_LLM1_FILEPATH.stem}_v2 to avoid overwriting ...")

    # If the original files exists but the v2 file doesn't, create the v2 file
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(static_prompt.render(context=context, question=question_1))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(dynamic_prompt.render(context=context, question=question_1))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_staticgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(static_prompt_geollama.render(context=context, question=question_2))
    with open(OUTPUT_LLM1_FILEPATH.parent / f"prompt_dynamicgeol_{OUTPUT_LLM1_FILEPATH.stem}_v2.txt", "w") as f:
        f.write(dynamic_prompt_geollama.render(context=context, question=question_2))
    
    df_resp_step1_all.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_step1_v2.csv"), encoding='utf-8', index=False)
    df_resp_step2_all.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent, f"{OUTPUT_LLM1_FILEPATH.stem}_step2_v2.csv"), encoding='utf-8', index=False)
    df_responses_error.to_csv(Path(OUTPUT_LLM1_FILEPATH.parent / f"errors_{OUTPUT_LLM1_FILEPATH.stem}_v2.csv"), encoding='utf-8', index=False)

else:
    print(f"Output file {Path(OUTPUT_LLM1_FILEPATH).stem} already exists. Skip saving to avoid overwriting ...")


# VOEW PROFILING RESUTLS
# scalene view --cli


11.117002752
0
0
Model calling via nvidia API:
Model calling via nvidia API:
11.117002752
Test mode is ON. Using only a small sample of documents for testing.


 ######## -------- Processing document [1/28]: Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent.md -------- ######## 


 ##### ------- Cleaning document -----------########
Cleaned markdown file already exists, loading file and proceeding with Ci impact extraction
Document cleaning took, 0.0 minutes
Chunking document...

Processing chunk no. 1 / 9 of document: Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent.md
Chunk text:  Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily
_
_
The Independent
0
0

STEP 1
0
0
[]
messages_list w tool: ['SYSTEM_PROMPT: \nYour name is LLM_1. You are an expert at structured data extraction.                        \nYour task is to analy

/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This 

### ---- Processing time for chunk 2 STEP 1: 0.3 minutes ---- ###
0
0

GEOLLAMA
Starting geoparsing
Time for geoparsing took 1.5
Removing duplicates, and keep countries in the geollama response
Enriching LLm 1 response with GeoLlama locations

STEP 2
KEEPING location info from step 1 for further improvement
11.117002752
calling LLM 2 took 0.11487342913945515 min
CREATED final LLM response (STEP 1 & 2) successfully

   Processing time for chunk 2 STEPs 1 & 2: 1.9 minutes

11.117002752
0

Processing chunk no. 3 / 9 of document: Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent.md
Chunk text:  Greece fires - live: Tourists in Rhodes and Corfu evacuated as Croatia also battles wildfires Major airport closed as wildfires now rage across Italy How long is the heatwave in Europe expected to last?
0
0

STEP 1
0
0
[Major airport]
messages_list w tool: ['SYSTEM_PROMPT: \nYour name is LLM_1. You are an expert at structured data extracti

/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This 

### ---- Processing time for chunk 3 STEP 1: 0.4 minutes ---- ###
0
0

GEOLLAMA
Starting geoparsing
Time for geoparsing took 0.9
Removing duplicates, and keep countries in the geollama response
Toponyms from geollama
 0      Italy
1    Croatia
2     Europe
3     Greece
4      Corfu
5     Rhodes
Name: name, dtype: str
Enriching LLm 1 response with GeoLlama locations

STEP 2
KEEPING location info from step 1 for further improvement
11.117002752
calling LLM 2 took 0.49312524795532225 min
CREATED final LLM response (STEP 1 & 2) successfully

   Processing time for chunk 3 STEPs 1 & 2: 1.7 minutes

11.117002752
0

Processing chunk no. 4 / 9 of document: Skoulding 2023 - Where are the fires in Italy today as temperatures rise to 47.6C on Sicily_ _ The Independent.md
Chunk text:  Italy - like much of southern Europe - has been hit by scorching temperatures bringing increased risk of fires and deaths.
Yesterday, temperatures rose to 47.6C in some parts of eastern Sicily, close to the record Eu

...

### ---- Processing time for chunk 10 STEP 1: 0.3 minutes ---- ###
0
0

GEOLLAMA
Starting geoparsing
Time for geoparsing took 1.0
Removing duplicates, and keep countries in the geollama response
Enriching LLm 1 response with GeoLlama locations

STEP 2
KEEPING location info from step 1 for further improvement
11.117002752
calling LLM 2 took 0.19582646290461223 min
CREATED final LLM response (STEP 1 & 2) successfully

   Processing time for chunk 10 STEPs 1 & 2: 1.5 minutes

11.117002752
0

Processing chunk no. 11 / 43 of document: Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events.md
Chunk text:  Winter storms are a challenge for energy meteorology in Europe for the threat to energy generation and transmission systems. This contribution focuses on the impact of Storm Xaver in northern Europe on 5-6 December 2013. There is a general literature overview of the infrastructure and societal impacts, and a more comprehensive reference list i

/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This 

### ---- Processing time for chunk 11 STEP 1: 0.3 minutes ---- ###
0
0

GEOLLAMA
Starting geoparsing
Time for geoparsing took 8.2
Removing duplicates, and keep countries in the geollama response
Enriching LLm 1 response with GeoLlama locations

STEP 2
KEEPING location info from step 1 for further improvement
11.117002752
calling LLM 2 took 0.03673113584518432 min
CREATED final LLM response (STEP 1 & 2) successfully

   Processing time for chunk 11 STEPs 1 & 2: 8.5 minutes

11.117002752
0

Processing chunk no. 12 / 43 of document: Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events.md
Chunk text:  This contribution follows on two previous studies of the Storm Britta in 2006 (Kettle, 2018) and Storm Tilo in 2007 (Kettle, 2019), which were also important for energy meteorology.
0
0

STEP 1
0
0
[]
messages_list w tool: ['SYSTEM_PROMPT: \nYour name is LLM_1. You are an expert at structured data extraction.                        \nYour task

/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This 

### ---- Processing time for chunk 16 STEP 1: 0.7 minutes ---- ###
0
0

GEOLLAMA
Starting geoparsing


In [ ]:
# he scalene extension is already loaded. To reload it, use:
#   %reload_ext scalene
# ERROR: Do not try to invoke `start` if you have not called Scalene using one of the methods
# in https://github.com/plasma-umass/scalene#using-scalene
# # (The most likely issue is that you need to run your code with `scalene`, not `python`).

In [ ]:
# df_resp = decoder_model_1.generate_response(
#     max_new_tokens = 2048,
#     user_prompt = user_prompt,
#     # user_dynamic_prompt: load_prompt_template,     
# )     
system_prompt=em.load_prompt_template(template_filename="gpt_system_prompt.txt")
system_prompt = system_prompt.render()
# user_prompt = user_prompt.render()
# user_dynamic_prompt = user_dynamic_prompt.render(
#     context=context,  # includes also df_ci_geo info
#     question=question,
# )

response = client.chat.completions.parse(
# with client.chat.completions.stream(
# response = client.responses.parse(  # use it with Pydantic List[str] for multi-output
    model=model_name, # "openai/gpt-oss-20b", #:fireworks-ai",
    temperature=0.0,
    top_p=0.01,
    reasoning_effort= "medium",
    # seed=42,
    #stream=False,
    # max_completion_tokens=1024,
    #max_tokens=1024,
    # prompt_cache_key
    # prompt_cache_retention= "24h",  # test past_key_value (iterative caching)
    messages=[
        {"role": "system", "content": f"{system_prompt}"},
        #{"role": "developer", "content": f"These are the single steps you should conduct for extracting the information about the impacts to critical infrastructure assets: {user_content}",},
        {"role": "user", "content": f"""{user_prompt}"""},
    ],
    #stop=["<extraction>"],
    response_format=Response,
    #include=["infrastructure_type", "location"],
)

try:
    # parsed_output = json.loads(response.output_text)
    parsed_output = json.loads(response.to_json())["choices"][0]["message"]["content"]
    # json.loads(response.to_json())["output"][1]["content"][0]["text"].split("ANSWER:")[1] 
    # print(parsed_output)    
    df_resp = pp.postprocess_response(parsed_output)
    
except Exception as e:
    print("Could not parse response as JSON:", e)
    parsed_output = None
    for output in response.choices[0]:
        if output[0] != "message":
            continue

        for item in output[1].content:
            if item.type == "refusal":
                # If the model refuses to respond, you will get a refusal message
                print(item.refusal)
                continue

            if not item.parsed:
                raise Exception("Could not parse response")
            
            print(item.parsed)


## TODO 


In [ ]:
resp.rpartition('"')[-3] + "]}"

In [ ]:
# print(f"Safety: saving responses (Step 1 + 2) for doc: {citation} ")
# df_resp_step1.to_csv(f"./interim_results/llm1_geollm_step1_{citation}.csv", encoding='utf-8', index=False)
# df_resp_step2.to_csv(f"llm1_geollm_step2_{citation}.csv", encoding='utf-8', index=False)

print(df_resp_step2.chunk_text[0])



## Doc. cleaning improve

In [ ]:
# c = """ 
# blublub title\n\n\n

# HERE IS NEW SUBSECTION:\n

# (large-scale) societal dis- ruptions dis - ruptions (Garschagen and Sandholz, 2018; Hallegatte et al., 2019; Fekete and Sandholz, 2021), empirical evidence on the impacts of extreme weather events on these systems is still

# Published by Copernicus Publications on behalf of the European Geosciences Union.

# E. E. Koks et al.: Flood impacts to infrastructure

# limited. This brief communication provides an overview of the observed ﬂood impacts to large-scale infrastructure sys- tems during the 2021 mid-July western European ﬂood event and how reconstruction of these large-scale systems has pro- 




# HERE IS NEW SUBSECTION

# severely damaged railway line (between the vil- lages of Spa and Pepinster) was reopened again on 3 Octo- ber 2021 (Rozendaal, 2021b). In the Netherlands, no large- scale damage has been reported to transport infrastructure. A few national highways were partly ﬂooded (e.g. the A76 in both directions) or brieﬂy closed (&lt; 3 d) because of the po- tential of ﬂooding. Most likely due to relative low-ﬂow ve- locities, damage to Dutch national road infrastructure was limited. Several railway sections were closed (e.g. the rail-

# way section between Maastricht and Liége) and some dam- age occurred to the railway infrastructure, in particular to the electronic “track circuit” devices and saturated railway em- bankments (Prorail, 2021).

# """
# #c = c.replace(r"\n", r" ")   # Isssue replaces also multipelinebreas eg before subsection
# c = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", c) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
# c = c.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
# # Matches \n not preceded or followed by \n
# # c = re.sub(r"(?<!\n)\n(?!\n)", r"\n", c)  # remove linebreaks only when the yoccured just once, but not for multiple linebreaks (e.g. before subsection)
# c = re.sub(r"\s+", " ", c)  # replace >1 whitespaces with single whitespace

# # c = c.replace(r"\w*- ", "\w*-", c)  # removes any word followed by "-"
# # c = re.sub(r"([^\s-])-\n([^\s-])", r"\1\2", c)  # remove hypen and linebreaks TODO test with koks sentences
# c = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", c)  # remove hypens in the middle of lines

# c 


# # 0090- some weird breaks-
# # And some long sentences 
# # which are not separated by dots but by line breaks and hyphens 



In [ ]:
# !uv pip install "unstructured[pdf]"  # unstructured  #langchain-unstructured #langchain-community
# # # !uv add langchain
# !uv lock
# !uv sync
# from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
# #from langchain.loaders import , UnstructuredFileLoader

# loader = DirectoryLoader(str(Path(DOCS_DIR)),  loader_cls=UnstructuredFileLoader, show_progress=True)
# pdf_docs = loader.load()
# # glob=glob("*.pdf"),
# print(f"Number of Documents: {len(pdf_docs)}")

# # convert the different layouts of the pdf files into unified markdown format incl. sub/section titles, tables, caption text etc
# for idx, doc in enumerate(pdf_docs, start=0):
#     print(doc)

In [ ]:
# pdf_filepath = Path("Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# # Path(DOCS_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")
# print(pdf_filepath)
# print("Remove reference section")

# # setup converter for PDF and markdown
# converter = DocumentConverter(
#     allowed_formats=[InputFormat.PDF, InputFormat.MD],
#     format_options={
#         InputFormat.PDF: FormatOption(
#             pipeline_cls=StandardPdfPipeline,
#             pipeline_options=pipeline_options,
#             backend=PyPdfiumDocumentBackend,
#         ),
#     },
# )
# pdf_text = converter.convert(pdf_filepath).document
    
# # loader = DoclingLoader("/beegfs/scratch/a-buch/_PROJECTS/data/text_sources/Lloyd's List 2024 - Port of Valencia reopens after devastating floods.pdf")  # use chunks from Docling.Loader
# # pdf_doc = loader.load()